In [ ]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [ ]:
#ablation study 



In [1]:
# ================================================================
# GRAPH-GROUNDED WIND-TRANSPORT TEST v3: three-way comparison to
# separate the STATIC distance component from the DYNAMIC wind
# component, since the full edge-weight formula confounds them
# (distance decay is the same at every hour, so it can dominate
# "top target" selection regardless of wind).
#
# Three target-selection rules, same top-vs-control paired test:
#   (1) full formula:     decay(dist) * wind_alignment * wind_speed
#   (2) distance-only:    decay(dist) alone (nearest eligible neighbor,
#                          static per source, ignores wind entirely)
#   (3) wind-only:        wind_alignment * wind_speed alone (no decay
#                          term, so distance cannot drive the pick)
# All window-averaged over the 36h post-onset period, 2016-2019.
# ================================================================
import numpy as np
import pandas as pd
import json
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

RHO_KM = 250.0
HYBRID_THRESHOLD_KM = 20.0
region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
eligible_mask = (region_dist_km > HYBRID_THRESHOLD_KM)
region_decay = np.exp(-region_dist_km / RHO_KM)

EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
FORWARD_WINDOW = 36

pm25_station = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order]
dt_index_full = pm25_station.index

year_mask = (dt_index_full.year >= 2016) & (dt_index_full.year <= 2019)
dt_index = dt_index_full[year_mask]
n_time = len(dt_index)
pm25_station_arr = pm25_station.to_numpy()[year_mask]

wdir_station = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index_full).to_numpy().astype(float)[year_mask]
wspeed_station = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index_full).to_numpy().astype(float)[year_mask]
wdir_sin_station = np.sin(np.radians(wdir_station))
wdir_cos_station = np.cos(np.radians(wdir_station))

def regional_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float64)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_mean(pm25_station_arr)
region_windspeed = regional_mean(wspeed_station)
region_wdir_sin = regional_mean(wdir_sin_station)
region_wdir_cos = regional_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

def onset_indices(label_col):
    lab = np.nan_to_num(label_col, nan=0.0)
    prev = np.concatenate([[0], lab[:-1]])
    return np.where((lab == 1) & (prev == 0))[0]

region_onsets = [onset_indices(region_episode_label[:, r]) for r in range(n_regions)]
onset_bool = np.zeros((n_time, n_regions), dtype=bool)
for r in range(n_regions):
    onset_bool[region_onsets[r], r] = True
onset_cumsum = np.cumsum(onset_bool, axis=0)

def any_onset_in_window(region_idx, t_start_excl, t_end_incl):
    t_end_incl = min(t_end_incl, n_time - 1)
    if t_end_incl <= t_start_excl:
        return False
    hi = onset_cumsum[t_end_incl, region_idx]
    lo = onset_cumsum[t_start_excl, region_idx] if t_start_excl >= 0 else 0
    return bool(hi - lo > 0)

print(f"date range: {dt_index[0]} to {dt_index[-1]}  n_time={n_time}")
print(f"total onset events (all regions): {sum(len(o) for o in region_onsets)}")

# ================================================================
# For each A-onset event, compute window-averaged scores under all
# three ranking rules, run top-vs-control test for each.
# ================================================================
mode_records = {"full": [], "distance_only": [], "wind_only": []}
top_target_counts = {"full": {}, "distance_only": {}, "wind_only": {}}

for a in range(n_regions):
    dist_score_a = np.where(eligible_mask[a, :], region_decay[a, :], np.nan)
    dist_score_a[a] = np.nan

    for t_A in region_onsets[a]:
        w_end = min(t_A + FORWARD_WINDOW, n_time)
        if w_end <= t_A:
            continue
        wbt_a_window = region_wind_blows_toward[t_A:w_end, a]
        speed_a_window = region_windspeed[t_A:w_end, a]
        if np.all(np.isnan(wbt_a_window)) or np.all(np.isnan(speed_a_window)):
            continue

        bearing_row = region_bearing[a, :]
        cos_align_TR = np.maximum(np.cos(np.radians(wbt_a_window[:, None] - bearing_row[None, :])), 0.0)
        wind_only_TR = cos_align_TR * speed_a_window[:, None]           # no decay term
        full_TR = region_decay[a, :][None, :] * wind_only_TR             # full formula

        avg_full = np.nanmean(full_TR, axis=0)
        avg_wind_only = np.nanmean(wind_only_TR, axis=0)

        avg_full = np.where(eligible_mask[a, :], avg_full, np.nan); avg_full[a] = np.nan
        avg_wind_only = np.where(eligible_mask[a, :], avg_wind_only, np.nan); avg_wind_only[a] = np.nan

        if np.all(np.isnan(avg_full)):
            continue

        scores = {"full": avg_full, "distance_only": dist_score_a, "wind_only": avg_wind_only}

        for mode, score_arr in scores.items():
            if np.all(np.isnan(score_arr)):
                continue
            top_target = int(np.nanargmax(score_arr))
            top_target_counts[mode][region_names[a]] = top_target_counts[mode].get(region_names[a], {})
            top_target_counts[mode][region_names[a]][region_names[top_target]] = \
                top_target_counts[mode][region_names[a]].get(region_names[top_target], 0) + 1

            hit_top = any_onset_in_window(top_target, t_A, t_A + FORWARD_WINDOW)
            other_hits = [any_onset_in_window(r, t_A, t_A + FORWARD_WINDOW)
                          for r in range(n_regions) if r != a and r != top_target and not np.isnan(score_arr[r])]
            control_hit_rate = float(np.mean(other_hits)) if other_hits else np.nan
            mode_records[mode].append({"source": region_names[a], "target_top": region_names[top_target],
                                        "hit_top": bool(hit_top), "control_hit_rate": control_hit_rate})

# ================================================================
# Results for each mode
# ================================================================
print(f"\n{'='*25} THREE-WAY COMPARISON {'='*25}")
summary = {}
for mode in ["full", "distance_only", "wind_only"]:
    recs = mode_records[mode]
    hit_top_arr = np.array([1.0 if r["hit_top"] else 0.0 for r in recs])
    control_arr = np.array([r["control_hit_rate"] for r in recs])
    valid = ~np.isnan(control_arr)
    hit_top_arr, control_arr = hit_top_arr[valid], control_arr[valid]
    diff = hit_top_arr - control_arr

    t_stat, p_two = stats.ttest_1samp(diff, 0.0)
    p_one_sided = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    w_stat, p_wilcoxon = stats.wilcoxon(diff)

    print(f"\n--- {mode} ---")
    print(f"n_events={len(diff)}")
    print(f"top-target hit rate:  {hit_top_arr.mean():.4f}")
    print(f"control rate:         {control_arr.mean():.4f}")
    print(f"lift:                 {diff.mean():+.4f}")
    print(f"paired t-test: t={t_stat:.3f}  p_one_sided={p_one_sided:.6f}")
    print(f"Wilcoxon: p={p_wilcoxon:.6f}")

    summary[mode] = {"n_events": len(diff), "top_rate": float(hit_top_arr.mean()),
                      "control_rate": float(control_arr.mean()), "lift": float(diff.mean()),
                      "t_stat": float(t_stat), "p_one_sided": float(p_one_sided), "p_wilcoxon": float(p_wilcoxon)}

print(f"\n{'='*25} SUMMARY TABLE {'='*25}")
print(f"{'mode':16s} {'top_rate':>10s} {'control':>10s} {'lift':>8s} {'p_one_sided':>12s}")
for mode in ["full", "distance_only", "wind_only"]:
    s = summary[mode]
    print(f"{mode:16s} {s['top_rate']:10.4f} {s['control_rate']:10.4f} {s['lift']:+8.4f} {s['p_one_sided']:12.6f}")

with open(f"{BASE}/ablation_wind_vs_distance_decomposition.json", "w") as f:
    json.dump({"summary": summary, "top_target_counts": top_target_counts}, f, indent=2)
print(f"\nsaved to {BASE}/ablation_wind_vs_distance_decomposition.json")


date range: 2016-01-01 00:00:00 to 2019-12-31 23:00:00  n_time=35064
total onset events (all regions): 708

========================= THREE-WAY COMPARISON =========================

--- full ---
n_events=708
top-target hit rate:  0.3884
control rate:         0.2978
lift:                 +0.0906
paired t-test: t=5.582  p_one_sided=0.000000
Wilcoxon: p=0.000001

--- distance_only ---
n_events=708
top-target hit rate:  0.3686
control rate:         0.2992
lift:                 +0.0695
paired t-test: t=4.148  p_one_sided=0.000019
Wilcoxon: p=0.000584

--- wind_only ---
n_events=708
top-target hit rate:  0.3362
control rate:         0.3014
lift:                 +0.0347
paired t-test: t=2.224  p_one_sided=0.013229
Wilcoxon: p=0.181794

========================= SUMMARY TABLE =========================
mode               top_rate    control     lift  p_one_sided
full                 0.3884     0.2978  +0.0906     0.000000
distance_only        0.3686     0.2992  +0.0695     0.000019
wind_only   

In [2]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated:
# Jeju gets ZERO neighbor features (no nearest-region lookups), matching
# the graph models where Jeju has no cross-region edges. Uses the tuned
# config found to be optimal on both the 2018 and within-COVID splits
# (lr=0.05, max_depth=4, l2=0.0), reused rather than re-searched.
# Split: train=2016-2017, val=2018, test=2019.
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

# ---- nearest-3-region lookup, EXCLUDING Jeju entirely (no neighbors for Jeju,
# and Jeju is never used as a neighbor for anyone else either, matching the
# graph models where all of Jeju's edges -- in and out -- are zeroed) ----
region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

# ================================================================
# Build features: single-region (mean+last per channel + static) +
# nearest-3-region mean+last PM2.5 (ZERO for Jeju, and Jeju excluded
# as a candidate neighbor for everyone else)
# ================================================================
buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)  # mean+last PM25 for up to 3 neighbors -> 6 values
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        # (Jeju: neighbors=[] -> neighbor_feats stays all zero)
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

# ---- reused tuned config (optimal on both 2018 and within-COVID splits) ----
BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

print(f"\n{'='*20} FULL 4-MODEL COMPARISON (2019 test, Jeju isolated) {'='*20}")
print(f"wind_graph (Jeju isolated, 10-seed):  AUC-PR=0.7064 +/- 0.0085")
print(f"no_graph (unaffected, 10-seed):       AUC-PR=0.6805 +/- 0.0025")
print(f"minimal_gru (unaffected, 10-seed):    AUC-PR=0.6743 +/- 0.0063")
print(f"HistGB+3nn (Jeju isolated, 1 run):    AUC-PR={test_aucpr:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_2019.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_2019.json")


train hours: 17544  val hours: 8760  test hours: 8760
rows: train=297041 val=147713 test=147713  (positive rate train=0.0370)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated) ====================
test AUC-ROC=0.9340  AUC-PR=0.6109
precision=0.5274  recall=0.5769  MCC=0.5250

==================== FULL 4-MODEL COMPARISON (2019 test, Jeju isolated) ====================
wind_graph (Jeju isolated, 10-seed):  AUC-PR=0.7064 +/- 0.0085
no_graph (unaffected, 10-seed):       AUC-PR=0.6805 +/- 0.0025
minimal_gru (unaffected, 10-seed):    AUC-PR=0.6743 +/- 0.0063
HistGB+3nn (Jeju isolated, 1 run):    AUC-PR=0.6109

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_2019.json


In [3]:
# ================================================================
# JEJU-ISOLATION ABLATION, 2018 SPLIT: fair-tuned wind_graph pipeline
# (train=2016, val=2017, test=2018), wind_graph ONLY, with every
# edge (station AND region level) touching Jeju forced to zero weight.
# Uses the 2018-optimal tuned config (lr=0.001, dropout=0.5, weight_decay=0.0005).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2016, val=2017, test=2018 ----
split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016): {TRAIN_MASK.sum()}  val hours (2017): {(split_id_per_hour==1).sum()}  test hours (2018): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

# ---- ISOLATE JEJU: zero out every edge (station AND region level) that
# touches Jeju, in either direction ----
jeju_idx = region_names.index("Jeju")
jeju_station_mask = (station_region_idx == jeju_idx)
station_touches_jeju = jeju_station_mask[src_idx] | jeju_station_mask[dst_idx]
station_wind_edge_weight[:, station_touches_jeju] = 0.0
region_touches_jeju = (r_src_idx == jeju_idx) | (r_dst_idx == jeju_idx)
region_wind_edge_weight[:, region_touches_jeju] = 0.0
print(f"zeroed {station_touches_jeju.sum()}/{len(station_touches_jeju)} station edges and "
      f"{region_touches_jeju.sum()}/{len(region_touches_jeju)} region edges touching Jeju")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds, wind_graph ONLY, Jeju isolated, 2018-optimal config
# ================================================================
N_SEEDS = 10
results = {"wind_graph_jeju_isolated": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph_jeju_isolated", wind_ewt_s, wind_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r, lr=0.001, dropout=0.5, weight_decay=0.0005)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)

        per_region = {}
        for r in range(n_regions):
            y_r = ytraj2[:, r, :].max(axis=-1)
            score_r = test_probs[:, r, :].max(axis=-1)
            if y_r.sum() == 0:
                continue
            per_region[region_names[r]] = {
                "aucpr": float(average_precision_score(y_r, score_r)),
                "aucroc": float(roc_auc_score(y_r, score_r)),
            }

        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}  "
              f"Jeju_AUCPR={per_region.get('Jeju', {}).get('aucpr', float('nan')):.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
                                     "per_region": per_region})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph_jeju_isolated"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph_jeju_isolated"]])
jeju_aucpr = np.array([r["per_region"].get("Jeju", {}).get("aucpr", np.nan) for r in results["wind_graph_jeju_isolated"]])
jeju_aucroc = np.array([r["per_region"].get("Jeju", {}).get("aucroc", np.nan) for r in results["wind_graph_jeju_isolated"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS (Jeju isolated, 2018 split) {'='*20}")
print(f"overall  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"overall  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"Jeju     AUC-PR:  mean={jeju_aucpr.mean():.4f}  std={jeju_aucpr.std():.4f}  values={jeju_aucpr.round(4)}")
print(f"Jeju     AUC-ROC: mean={jeju_aucroc.mean():.4f}  std={jeju_aucroc.std():.4f}  values={jeju_aucroc.round(4)}")

print(f"\n{'='*20} COMPARE TO CONNECTED BASELINE (2018 split) {'='*20}")
print(f"overall wind_graph (connected, 10-seed mean):  AUC-PR=0.4628")
print(f"Jeju    wind_graph (ISOLATED, {N_SEEDS}-seed mean): AUC-PR={jeju_aucpr.mean():.4f}  AUC-ROC={jeju_aucroc.mean():.4f}")

print(f"\n{'='*20} PER-REGION AUC-PR, AVERAGED ACROSS {N_SEEDS} SEEDS {'='*20}")
for r in range(n_regions):
    name = region_names[r]
    vals = np.array([res["per_region"].get(name, {}).get("aucpr", np.nan) for res in results["wind_graph_jeju_isolated"]], dtype=float)
    print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")

with open(f"{BASE}/wind_graph_jeju_isolated_2018split_10seed.json", "w") as f:
    json.dump({"wind_graph_jeju_isolated": results["wind_graph_jeju_isolated"],
               "overall_aucpr_mean": float(wind_aucpr.mean()), "overall_aucpr_std": float(wind_aucpr.std()),
               "jeju_aucpr_mean": float(jeju_aucpr.mean()), "jeju_aucpr_std": float(jeju_aucpr.std())}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_jeju_isolated_2018split_10seed.json")


device=mps
train hours (2016): 8784  val hours (2017): 8760  test hours (2018): 8760
zeroed 140/21414 station edges and 32/272 region edges touching Jeju
windows: train=8713 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph_jeju_isolated ==========
  trained in 459s  val_AUCPR=0.4326  test_AUCROC=0.9287  test_AUCPR=0.4577  Jeju_AUCPR=0.1326

========== seed=1  model=wind_graph_jeju_isolated ==========
  trained in 452s  val_AUCPR=0.4557  test_AUCROC=0.9314  test_AUCPR=0.4793  Jeju_AUCPR=0.1536

========== seed=2  model=wind_graph_jeju_isolated ==========
  trained in 457s  val_AUCPR=0.4315  test_AUCROC=0.9292  test_AUCPR=0.4373  Jeju_AUCPR=0.1073

========== seed=3  model=wind_graph_jeju_isolated ==========
  trained in 459s  val_AUCPR=0.4425  test_AUCROC=0.9289  test_AUCPR=0.4680  Jeju_AUCPR=0.1683

========== seed=4  model=wind_graph_jeju_isolated ==========
  trained in 458s  val_AUCPR=0.4568  test_AUCROC=0.9319  test_AUCPR=0.4814  Jeju_

In [4]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated,
# 2018 split (train=2016, val=2017, test=2018). Reuses the tuned
# config (lr=0.05, max_depth=4, l2=0.0), same as found optimal on
# this exact split's single-region tree baseline and on within-COVID.
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()

# ---- SPLIT: train=2016, val=2017, test=2018 ----
split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated, 2018 split) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_2018.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_2018.json")


train hours: 8784  val hours: 8760  test hours: 8760
rows: train=148121 val=147713 test=147713  (positive rate train=0.0368)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated, 2018 split) ====================
test AUC-ROC=0.9110  AUC-PR=0.4013
precision=0.3831  recall=0.4414  MCC=0.3855

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_2018.json


In [ ]:
#2020-2021 split. 



In [1]:
# ================================================================
# JEJU-ISOLATION ABLATION, WITHIN-COVID-REGIME SPLIT:
# train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021. wind_graph ONLY,
# every edge (station AND region level) touching Jeju forced to zero
# weight. Uses the tuned config for this split
# (lr=0.001, dropout=0.5, weight_decay=0.0005).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2020): {TRAIN_MASK.sum()}  val hours (H1-2021): {(split_id_per_hour==1).sum()}  test hours (H2-2021): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

# ---- ISOLATE JEJU: zero out every edge (station AND region level) that
# touches Jeju, in either direction ----
jeju_idx = region_names.index("Jeju")
jeju_station_mask = (station_region_idx == jeju_idx)
station_touches_jeju = jeju_station_mask[src_idx] | jeju_station_mask[dst_idx]
station_wind_edge_weight[:, station_touches_jeju] = 0.0
region_touches_jeju = (r_src_idx == jeju_idx) | (r_dst_idx == jeju_idx)
region_wind_edge_weight[:, region_touches_jeju] = 0.0
print(f"zeroed {station_touches_jeju.sum()}/{len(station_touches_jeju)} station edges and "
      f"{region_touches_jeju.sum()}/{len(region_touches_jeju)} region edges touching Jeju")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds, wind_graph ONLY, Jeju isolated, within-COVID config
# ================================================================
N_SEEDS = 10
results = {"wind_graph_jeju_isolated": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph_jeju_isolated", wind_ewt_s, wind_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r, lr=0.001, dropout=0.5, weight_decay=0.0005)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)

        per_region = {}
        for r in range(n_regions):
            y_r = ytraj2[:, r, :].max(axis=-1)
            score_r = test_probs[:, r, :].max(axis=-1)
            if y_r.sum() == 0:
                continue
            per_region[region_names[r]] = {
                "aucpr": float(average_precision_score(y_r, score_r)),
                "aucroc": float(roc_auc_score(y_r, score_r)),
            }

        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}  "
              f"Jeju_AUCPR={per_region.get('Jeju', {}).get('aucpr', float('nan')):.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
                                     "per_region": per_region})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph_jeju_isolated"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph_jeju_isolated"]])
jeju_aucpr = np.array([r["per_region"].get("Jeju", {}).get("aucpr", np.nan) for r in results["wind_graph_jeju_isolated"]])
jeju_aucroc = np.array([r["per_region"].get("Jeju", {}).get("aucroc", np.nan) for r in results["wind_graph_jeju_isolated"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS (Jeju isolated, within-COVID split) {'='*20}")
print(f"overall  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"overall  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"Jeju     AUC-PR:  mean={jeju_aucpr.mean():.4f}  std={jeju_aucpr.std():.4f}  values={jeju_aucpr.round(4)}")
print(f"Jeju     AUC-ROC: mean={jeju_aucroc.mean():.4f}  std={jeju_aucroc.std():.4f}  values={jeju_aucroc.round(4)}")

print(f"\n{'='*20} COMPARE TO CONNECTED BASELINE (within-COVID split) {'='*20}")
print(f"overall wind_graph (connected, 10-seed mean):  AUC-PR=0.4904")
print(f"Jeju    wind_graph (ISOLATED, {N_SEEDS}-seed mean): AUC-PR={jeju_aucpr.mean():.4f}  AUC-ROC={jeju_aucroc.mean():.4f}")

print(f"\n{'='*20} PER-REGION AUC-PR, AVERAGED ACROSS {N_SEEDS} SEEDS {'='*20}")
for r in range(n_regions):
    name = region_names[r]
    vals = np.array([res["per_region"].get(name, {}).get("aucpr", np.nan) for res in results["wind_graph_jeju_isolated"]], dtype=float)
    print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")

with open(f"{BASE}/wind_graph_jeju_isolated_covidsplit_10seed.json", "w") as f:
    json.dump({"wind_graph_jeju_isolated": results["wind_graph_jeju_isolated"],
               "overall_aucpr_mean": float(wind_aucpr.mean()), "overall_aucpr_std": float(wind_aucpr.std()),
               "jeju_aucpr_mean": float(jeju_aucpr.mean()), "jeju_aucpr_std": float(jeju_aucpr.std())}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_jeju_isolated_covidsplit_10seed.json")


device=mps
train hours (2020): 8784  val hours (H1-2021): 4344  test hours (H2-2021): 4416
zeroed 140/21414 station edges and 32/272 region edges touching Jeju
windows: train=8713 val=4273 test=4345
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph_jeju_isolated ==========
  trained in 323s  val_AUCPR=0.3057  test_AUCROC=0.9676  test_AUCPR=0.5383  Jeju_AUCPR=nan

========== seed=1  model=wind_graph_jeju_isolated ==========
  trained in 315s  val_AUCPR=0.2805  test_AUCROC=0.9518  test_AUCPR=0.4272  Jeju_AUCPR=nan

========== seed=2  model=wind_graph_jeju_isolated ==========
  trained in 313s  val_AUCPR=0.3269  test_AUCROC=0.9497  test_AUCPR=0.4411  Jeju_AUCPR=nan

========== seed=3  model=wind_graph_jeju_isolated ==========
  trained in 314s  val_AUCPR=0.3218  test_AUCROC=0.9572  test_AUCPR=0.4847  Jeju_AUCPR=nan

========== seed=4  model=wind_graph_jeju_isolated ==========
  trained in 313s  val_AUCPR=0.3084  test_AUCROC=0.9510  test_AUCPR=0.5059  Jeju_AUCPR=

/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_2702/1661266546.py:515: RuntimeWarning: Mean of empty slice
  print(f"  {name:10s}  AUC-PR mean={np.nanmean(vals):.4f}  std={np.nanstd(vals):.4f}")
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [1]:
# ================================================================
# TREE BASELINE (HistGB + nearest-3-region features), Jeju isolated,
# within-COVID-regime split (train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021).
# Reuses the tuned config (lr=0.05, max_depth=4, l2=0.0).
# ================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve
import json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
jeju_idx = region_names.index("Jeju")
nearest3 = {}
for r in range(n_regions):
    if r == jeju_idx:
        nearest3[r] = []  # Jeju: no neighbors at all
        continue
    order = np.argsort(region_dist_km[r])
    order = [o for o in order if o != r and o != jeju_idx]  # exclude self and Jeju as a candidate neighbor
    nearest3[r] = order[:3]

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(time_panels["PM25"].to_numpy())

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
doy = dt_index.dayofyear.to_numpy().astype(float)
season_sin = np.tile(np.sin(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))
season_cos = np.tile(np.cos(2 * np.pi * doy / 365.25)[:, None], (1, n_stations))

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, np.sin(np.radians(wind_dir_arr)), np.cos(np.radians(wind_dir_arr)), blh_arr, season_sin, season_cos], axis=-1)

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev

print(f"train hours: {(split_id_per_hour==0).sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    pm25_col = TIME_FEATS.index("PM25")
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        neighbor_feats = np.zeros(6, dtype=np.float32)
        neighbors = nearest3[r]
        for ni, nb in enumerate(neighbors):
            neighbor_feats[ni] = window_mean[nb, pm25_col]
            neighbor_feats[3 + ni] = window_last[nb, pm25_col]
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r], neighbor_feats])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

BEST_CFG = {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0}
print(f"using reused config: {BEST_CFG}")

model = HistGradientBoostingClassifier(
    max_iter=300, **BEST_CFG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = model.predict_proba(X1)[:, 1]
test_score = model.predict_proba(X2)[:, 1]
test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]
pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)

print(f"\n{'='*20} TREE BASELINE (nearest-3-region features, Jeju isolated, within-COVID split) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"precision={precision:.4f}  recall={recall:.4f}  MCC={mcc:.4f}")

with open(f"{BASE}/histgb_3nn_jeju_isolated_covid.json", "w") as f:
    json.dump({"config": BEST_CFG, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "precision": float(precision), "recall": float(recall), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_3nn_jeju_isolated_covid.json")


train hours: 8784  val hours: 4344  test hours: 4416
rows: train=148121 val=72641 test=73865  (positive rate train=0.0134)
using reused config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}

==================== TREE BASELINE (nearest-3-region features, Jeju isolated, within-COVID split) ====================
test AUC-ROC=0.9602  AUC-PR=0.1584
precision=0.1740  recall=0.4631  MCC=0.2710

saved to /Users/drewbaldwin/PM2_5 Research/histgb_3nn_jeju_isolated_covid.json


In [1]:
#obtain key metrics at different forecast horizons. 

# ================================================================
# HORIZON-STRATIFIED EVALUATION (12h / 24h / 36h), 2019 test year.
# wind_graph: loaded from saved_wind_graph_2019/ (exact reproduction).
# no_graph, minimal_gru: retrained fresh (single seed=0, fair-tuned configs).
# Metrics per horizon: AUC-ROC, AUC-PR, precision, recall (at val
# F1-optimal threshold), and mean detection lead time (hours).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    """No graph convolution at all -- per-station Linear -> GRU."""
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges (needed for wind_graph inference and no_graph training) ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

# ---- normalization: use the SAVED wind_graph normalization for consistency ----
norm = np.load(f"{BASE}/saved_wind_graph_2019/normalization.npz")
t_mean, t_std = norm["t_mean"], norm["t_std"]
s_mean, s_std = norm["s_mean"], norm["s_std"]
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

# ================================================================
# MODEL 1: wind_graph -- load saved 2019 model, no retraining
# ================================================================
print(f"\n{'#'*15} loading saved wind_graph 2019 model {'#'*15}")
wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

wind_val_probs = predict_traj_probs_graph(wind_model, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
wind_test_probs = predict_traj_probs_graph(wind_model, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# MODEL 2: no_graph -- fresh training, fair-tuned config
# ================================================================
print(f"\n{'#'*15} training no_graph (seed=0, tuned config) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
lr, dropout, weight_decay = 0.001, 0.5, 0.0005
p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_graph(p1, zero_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch_graph(p1, zero_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
nograph_model = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
del p1; gc.collect()

opt2 = torch.optim.Adam(nograph_model.parameters(), lr=lr, weight_decay=weight_decay)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    run_p2_traj_epoch_graph(nograph_model, zero_ewt_s, zero_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
    run_p2_traj_epoch_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
    vp = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(nograph_model.state_dict())
nograph_model.load_state_dict(best_state)
nograph_model.eval()
print(f"no_graph val_aucpr={best_val_aucpr:.4f}")

nograph_val_probs = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X1_t, starts1_t)
nograph_test_probs = predict_traj_probs_graph(nograph_model, zero_ewt_s, zero_ewt_r, X2_t, starts2_t)
del nograph_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# MODEL 3: minimal_gru -- fresh training, fair-tuned config
# ================================================================
print(f"\n{'#'*15} training minimal_gru (seed=0, tuned config) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
lr, dropout, weight_decay = 0.003, 0.3, 0.0005
p1 = StationGRU(n_feats, dropout).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
    vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_lin)
minimal_model = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
del p1; gc.collect()

opt2 = torch.optim.Adam(minimal_model.parameters(), lr=lr, weight_decay=weight_decay)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    run_p2_traj_epoch_gru(minimal_model, X0_t, ytraj0_t, opt2, True)
    run_p2_traj_epoch_gru(minimal_model, X1_t, ytraj1_t, opt2, False)
    vp = predict_traj_probs_gru(minimal_model, X1_t)
    va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(minimal_model.state_dict())
minimal_model.load_state_dict(best_state)
minimal_model.eval()
print(f"minimal_gru val_aucpr={best_val_aucpr:.4f}")

minimal_val_probs = predict_traj_probs_gru(minimal_model, X1_t)
minimal_test_probs = predict_traj_probs_gru(minimal_model, X2_t)
del minimal_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# HORIZON-STRATIFIED METRICS for the 3 trajectory models
# ================================================================
def horizon_metrics(val_probs, val_labels, test_probs, test_labels, H, model_name):
    val_score_H = val_probs[..., :H].max(axis=-1).reshape(-1)
    val_label_H = val_labels[..., :H].max(axis=-1).reshape(-1)
    test_score_H = test_probs[..., :H].max(axis=-1).reshape(-1)
    test_label_H = test_labels[..., :H].max(axis=-1).reshape(-1)

    aucroc = roc_auc_score(test_label_H, test_score_H)
    aucpr = average_precision_score(test_label_H, test_score_H)

    fpr, tpr, thresholds = roc_curve(val_label_H, val_score_H)
    f1s = []
    for th in thresholds:
        pred = (val_score_H >= th).astype(int)
        p = precision_score(val_label_H, pred, zero_division=0)
        r = recall_score(val_label_H, pred, zero_division=0)
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    best_thresh = thresholds[np.argmax(f1s)]

    pred_test = (test_score_H >= best_thresh).astype(int)
    precision = precision_score(test_label_H, pred_test, zero_division=0)
    recall = recall_score(test_label_H, pred_test, zero_division=0)

    n_win, n_reg, _ = test_labels.shape
    lead_times = []
    n_true_onsets = 0
    for w in range(n_win):
        for r in range(n_reg):
            lab = test_labels[w, r, :H]
            if lab.max() == 0:
                continue
            n_true_onsets += 1
            onset_h = int(np.argmax(lab))
            prob = test_probs[w, r, :H]
            alarm_idx = np.where(prob >= best_thresh)[0]
            alarm_idx = alarm_idx[alarm_idx <= onset_h]
            if len(alarm_idx) > 0:
                alarm_h = alarm_idx[0]
                lead_times.append(onset_h - alarm_h)
    mean_lead = float(np.mean(lead_times)) if lead_times else float("nan")
    detect_rate = len(lead_times) / n_true_onsets if n_true_onsets else float("nan")

    print(f"  [{model_name}] H={H}h  AUC-ROC={aucroc:.4f}  AUC-PR={aucpr:.4f}  "
          f"precision={precision:.4f}  recall={recall:.4f}  "
          f"mean_lead={mean_lead:.2f}h  detect_rate={detect_rate:.2%}  n_onsets={n_true_onsets}")
    return {"H": H, "aucroc": float(aucroc), "aucpr": float(aucpr), "precision": float(precision),
            "recall": float(recall), "mean_lead_hours": mean_lead, "detect_rate": detect_rate,
            "n_true_onsets": n_true_onsets, "threshold": float(best_thresh)}

HORIZONS = [12, 24, 36]
results = {"wind_graph": [], "no_graph": [], "minimal_gru": []}

print(f"\n{'='*20} HORIZON-STRATIFIED METRICS {'='*20}")
for H in HORIZONS:
    results["wind_graph"].append(horizon_metrics(wind_val_probs, ytraj1, wind_test_probs, ytraj2, H, "wind_graph"))
for H in HORIZONS:
    results["no_graph"].append(horizon_metrics(nograph_val_probs, ytraj1, nograph_test_probs, ytraj2, H, "no_graph"))
for H in HORIZONS:
    results["minimal_gru"].append(horizon_metrics(minimal_val_probs, ytraj1, minimal_test_probs, ytraj2, H, "minimal_gru"))

with open(f"{BASE}/horizon_stratified_2019_graph_gru.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/horizon_stratified_2019_graph_gru.json")
print("Now run the separate tree-baseline script to complete the comparison.")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689

############### loading saved wind_graph 2019 model ###############

############### training no_graph (seed=0, tuned config) ###############
no_graph val_aucpr=0.4681

############### training minimal_gru (seed=0, tuned config) ###############
minimal_gru val_aucpr=0.4694

==================== HORIZON-STRATIFIED METRICS ====================
  [wind_graph] H=12h  AUC-ROC=0.9862  AUC-PR=0.8113  precision=0.7879  recall=0.6780  mean_lead=1.13h  detect_rate=67.63%  n_onsets=4745
  [wind_graph] H=24h  AUC-ROC=0.9741  AUC-PR=0.7686  precision=0.7114  recall=0.6811  mean_lead=3.19h  detect_rate=66.56%  n_onsets=6426
  [wind_graph] H=36h  AUC-ROC=0.9580  AUC-PR=0.7154  precision=0.7129  recall=0.5960  mean_lead=4.27h  detect_rate=58.01%  n_onsets=7929
  [no_graph] H=12h  AUC-ROC=0.9792  AUC-PR=0.7842  precision=0.7743  recall=0.6499  mean_lead=0.98h  detect_rate=

In [1]:
#more seeds to remove variation. 


# ================================================================
# HORIZON-STRATIFIED EVALUATION (12h / 24h / 36h), 2019 test year,
# 5 SEEDS for wind_graph, no_graph, minimal_gru (all trained fresh
# with their respective tuned configs, for full methodological
# consistency across the three models -- no mixing loaded + trained).
# Reports mean +/- std across seeds per horizon per model.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def horizon_metrics(val_probs, val_labels, test_probs, test_labels, H):
    val_score_H = val_probs[..., :H].max(axis=-1).reshape(-1)
    val_label_H = val_labels[..., :H].max(axis=-1).reshape(-1)
    test_score_H = test_probs[..., :H].max(axis=-1).reshape(-1)
    test_label_H = test_labels[..., :H].max(axis=-1).reshape(-1)

    aucroc = roc_auc_score(test_label_H, test_score_H)
    aucpr = average_precision_score(test_label_H, test_score_H)

    fpr, tpr, thresholds = roc_curve(val_label_H, val_score_H)
    f1s = []
    for th in thresholds:
        pred = (val_score_H >= th).astype(int)
        p = precision_score(val_label_H, pred, zero_division=0)
        r = recall_score(val_label_H, pred, zero_division=0)
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    best_thresh = thresholds[np.argmax(f1s)]

    pred_test = (test_score_H >= best_thresh).astype(int)
    precision = precision_score(test_label_H, pred_test, zero_division=0)
    recall = recall_score(test_label_H, pred_test, zero_division=0)

    n_win, n_reg, _ = test_labels.shape
    lead_times = []
    n_true_onsets = 0
    for w in range(n_win):
        for r in range(n_reg):
            lab = test_labels[w, r, :H]
            if lab.max() == 0:
                continue
            n_true_onsets += 1
            onset_h = int(np.argmax(lab))
            prob = test_probs[w, r, :H]
            alarm_idx = np.where(prob >= best_thresh)[0]
            alarm_idx = alarm_idx[alarm_idx <= onset_h]
            if len(alarm_idx) > 0:
                lead_times.append(onset_h - alarm_idx[0])
    mean_lead = float(np.mean(lead_times)) if lead_times else float("nan")
    detect_rate = len(lead_times) / n_true_onsets if n_true_onsets else float("nan")

    return {"aucroc": float(aucroc), "aucpr": float(aucpr), "precision": float(precision),
            "recall": float(recall), "mean_lead_hours": mean_lead, "detect_rate": detect_rate}

HORIZONS = [12, 24, 36]
N_SEEDS = 5
CONFIGS = {
    "wind_graph": {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005, "ewt_s": wind_ewt_s, "ewt_r": wind_ewt_r},
    "no_graph": {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005, "ewt_s": zero_ewt_s, "ewt_r": zero_ewt_r},
}
MINIMAL_CFG = {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005}

all_results = {"wind_graph": {H: [] for H in HORIZONS}, "no_graph": {H: [] for H in HORIZONS}, "minimal_gru": {H: [] for H in HORIZONS}}

for seed in range(N_SEEDS):
    for model_name, cfg in CONFIGS.items():
        print(f"\n{'#'*10} seed={seed}  model={model_name} {'#'*10}")
        torch.manual_seed(seed); np.random.seed(seed)
        lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
        ewt_s, ewt_r = cfg["ewt_s"], cfg["ewt_r"]
        p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
        opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
        best_p1_val, best_p1_state = float("inf"), None
        for epoch in range(1, MAX_EPOCHS_P1 + 1):
            run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
            vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
            if vl < best_p1_val:
                best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
        p1.load_state_dict(best_p1_state)
        encoder_copy = copy.deepcopy(p1.station_conv)
        model = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
        del p1; gc.collect()

        opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
        best_val_aucpr, best_state = -1.0, None
        for epoch in range(1, MAX_EPOCHS_P2 + 1):
            run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
            run_p2_traj_epoch_graph(model, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
            vp = predict_traj_probs_graph(model, ewt_s, ewt_r, X1_t, starts1_t)
            va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
            if va > best_val_aucpr:
                best_val_aucpr, best_state = va, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        model.eval()
        print(f"  val_aucpr={best_val_aucpr:.4f}")

        val_probs = predict_traj_probs_graph(model, ewt_s, ewt_r, X1_t, starts1_t)
        test_probs = predict_traj_probs_graph(model, ewt_s, ewt_r, X2_t, starts2_t)
        del model; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

        for H in HORIZONS:
            m = horizon_metrics(val_probs, ytraj1, test_probs, ytraj2, H)
            all_results[model_name][H].append(m)
            print(f"    H={H}h  AUC-PR={m['aucpr']:.4f}  AUC-ROC={m['aucroc']:.4f}  "
                  f"precision={m['precision']:.4f}  recall={m['recall']:.4f}  lead={m['mean_lead_hours']:.2f}h")

    # minimal_gru
    print(f"\n{'#'*10} seed={seed}  model=minimal_gru {'#'*10}")
    torch.manual_seed(seed); np.random.seed(seed)
    lr, dropout, weight_decay = MINIMAL_CFG["lr"], MINIMAL_CFG["dropout"], MINIMAL_CFG["weight_decay"]
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    model = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()

    opt2 = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch_gru(model, X0_t, ytraj0_t, opt2, True)
        run_p2_traj_epoch_gru(model, X1_t, ytraj1_t, opt2, False)
        vp = predict_traj_probs_gru(model, X1_t)
        va = average_precision_score(y_val_agg, vp.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    print(f"  val_aucpr={best_val_aucpr:.4f}")

    val_probs = predict_traj_probs_gru(model, X1_t)
    test_probs = predict_traj_probs_gru(model, X2_t)
    del model; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    for H in HORIZONS:
        m = horizon_metrics(val_probs, ytraj1, test_probs, ytraj2, H)
        all_results["minimal_gru"][H].append(m)
        print(f"    H={H}h  AUC-PR={m['aucpr']:.4f}  AUC-ROC={m['aucroc']:.4f}  "
              f"precision={m['precision']:.4f}  recall={m['recall']:.4f}  lead={m['mean_lead_hours']:.2f}h")

# ================================================================
# AGGREGATE: mean +/- std across 5 seeds, per model per horizon
# ================================================================
print(f"\n{'='*20} AGGREGATED ACROSS {N_SEEDS} SEEDS {'='*20}")
summary = {}
for model_name in ["wind_graph", "no_graph", "minimal_gru"]:
    summary[model_name] = {}
    for H in HORIZONS:
        recs = all_results[model_name][H]
        agg = {k: (float(np.mean([r[k] for r in recs])), float(np.std([r[k] for r in recs])))
               for k in ["aucroc", "aucpr", "precision", "recall", "mean_lead_hours", "detect_rate"]}
        summary[model_name][H] = agg
        print(f"[{model_name}] H={H}h  "
              f"AUC-PR={agg['aucpr'][0]:.4f}+/-{agg['aucpr'][1]:.4f}  "
              f"AUC-ROC={agg['aucroc'][0]:.4f}+/-{agg['aucroc'][1]:.4f}  "
              f"precision={agg['precision'][0]:.4f}+/-{agg['precision'][1]:.4f}  "
              f"recall={agg['recall'][0]:.4f}+/-{agg['recall'][1]:.4f}  "
              f"lead={agg['mean_lead_hours'][0]:.2f}+/-{agg['mean_lead_hours'][1]:.2f}h")

with open(f"{BASE}/horizon_stratified_2019_5seed.json", "w") as f:
    json.dump({"per_seed": all_results, "summary": summary}, f, indent=2)
print(f"\nsaved to {BASE}/horizon_stratified_2019_5seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689

########## seed=0  model=wind_graph ##########
  val_aucpr=0.5071
    H=12h  AUC-PR=0.8113  AUC-ROC=0.9860  precision=0.7942  recall=0.6643  lead=1.10h
    H=24h  AUC-PR=0.7693  AUC-ROC=0.9738  precision=0.7460  recall=0.6555  lead=3.01h
    H=36h  AUC-PR=0.7161  AUC-ROC=0.9575  precision=0.6889  recall=0.6215  lead=4.48h

########## seed=0  model=no_graph ##########
  val_aucpr=0.4681
    H=12h  AUC-PR=0.7842  AUC-ROC=0.9792  precision=0.7743  recall=0.6499  lead=0.98h
    H=24h  AUC-PR=0.7305  AUC-ROC=0.9639  precision=0.7422  recall=0.6078  lead=2.63h
    H=36h  AUC-PR=0.6777  AUC-ROC=0.9474  precision=0.6098  recall=0.6174  lead=4.56h

########## seed=0  model=minimal_gru ##########
  val_aucpr=0.4694
    H=12h  AUC-PR=0.7883  AUC-ROC=0.9809  precision=0.7488  recall=0.6752  lead=0.86h
    H=24h  AUC-PR=0.7374  AUC-ROC=0.9661  precision=0.7294  recall=

In [1]:
# ================================================================
# WATCH / WARNING TIERED ALERT SYSTEM, built on the already-validated
# wind_graph 2019 model's hourly probability trajectory (36h horizon).
# WATCH: sensitive threshold (precision floor ~0.25) -- "conditions
#   favorable for a sustained episode," earliest possible notice.
# WARNING: the established F1-optimal threshold -- "episode imminent,"
#   higher confidence, shorter notice.
# Reuses the saved 2019 model (no retraining).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, gc, json
from sklearn.metrics import precision_score, recall_score, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges (same formula as headline model) ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

norm = np.load(f"{BASE}/saved_wind_graph_2019/normalization.npz")
t_mean, t_std = norm["t_mean"], norm["t_std"]
s_mean, s_std = norm["s_mean"], norm["s_std"]
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    ytrajl.append(traj.T)
    sl.append(t)

X1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: val={len(X1)} test={len(X2)}")

X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts1, starts2))
gc.collect()

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} loading saved wind_graph 2019 model {'#'*15}")
wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

val_probs = predict_traj_probs(wind_model, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
test_probs = predict_traj_probs(wind_model, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# THRESHOLD SELECTION on VAL, full 36h horizon:
# WARNING: F1-optimal (same convention as the rest of the paper)
# WATCH: lowest threshold with precision >= 0.25 (sensitive, wide-net)
# ================================================================
val_score = val_probs.max(axis=-1).reshape(-1)
val_label = ytraj1.max(axis=-1).reshape(-1)

fpr, tpr, thresholds = roc_curve(val_label, val_score)
f1s, precisions = [], []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(val_label, pred, zero_division=0)
    r = recall_score(val_label, pred, zero_division=0)
    precisions.append(p)
    f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
precisions = np.array(precisions)

warning_thresh = thresholds[np.argmax(f1s)]
PRECISION_FLOOR = 0.25
eligible = np.where(precisions >= PRECISION_FLOOR)[0]
watch_thresh = thresholds[eligible[np.argmin(thresholds[eligible])]] if len(eligible) > 0 else warning_thresh

print(f"\nWARNING threshold (F1-optimal): {warning_thresh:.4f}")
print(f"WATCH threshold (precision>={PRECISION_FLOOR}): {watch_thresh:.4f}")

# ================================================================
# EVALUATE ON TEST: for each true onset, find first hour crossing
# WATCH and first hour crossing WARNING; report lead times, detection
# rates, and the "escalation gap" (extra lead time WATCH buys over
# WARNING for the same event).
# ================================================================
n_win, n_reg, H = ytraj2.shape
watch_leads, warning_leads, gaps = [], [], []
n_true_onsets = 0
n_watch_detected = 0
n_warning_detected = 0

for w in range(n_win):
    for r in range(n_reg):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        n_true_onsets += 1
        onset_h = int(np.argmax(lab))
        prob = test_probs[w, r, :]

        watch_idx = np.where(prob >= watch_thresh)[0]
        watch_idx = watch_idx[watch_idx <= onset_h]
        warning_idx = np.where(prob >= warning_thresh)[0]
        warning_idx = warning_idx[warning_idx <= onset_h]

        watch_h = watch_idx[0] if len(watch_idx) > 0 else None
        warning_h = warning_idx[0] if len(warning_idx) > 0 else None

        if watch_h is not None:
            n_watch_detected += 1
            watch_leads.append(onset_h - watch_h)
        if warning_h is not None:
            n_warning_detected += 1
            warning_leads.append(onset_h - warning_h)
        if watch_h is not None and warning_h is not None:
            gaps.append((onset_h - watch_h) - (onset_h - warning_h))  # = warning_h - watch_h

# ---- false-alarm rate: fraction of NON-onset windows that ever cross each threshold ----
non_onset_mask = ytraj2.max(axis=-1) == 0  # (n_win, n_reg)
watch_false = np.mean((test_probs >= watch_thresh).any(axis=-1)[non_onset_mask])
warning_false = np.mean((test_probs >= warning_thresh).any(axis=-1)[non_onset_mask])

print(f"\n{'='*20} WATCH / WARNING TIERED SYSTEM, 2019 TEST (full 36h horizon) {'='*20}")
print(f"total true onsets: {n_true_onsets}")
print(f"\nWATCH   (thresh={watch_thresh:.4f}):  detected {n_watch_detected}/{n_true_onsets} ({n_watch_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(watch_leads):.2f}h  median lead={np.median(watch_leads):.1f}h  false-alarm rate={watch_false:.1%}")
print(f"WARNING (thresh={warning_thresh:.4f}):  detected {n_warning_detected}/{n_true_onsets} ({n_warning_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(warning_leads):.2f}h  median lead={np.median(warning_leads):.1f}h  false-alarm rate={warning_false:.1%}")
print(f"\nESCALATION GAP (extra hours WATCH buys over WARNING, for events caught by both, n={len(gaps)}):")
print(f"  mean={np.mean(gaps):.2f}h  median={np.median(gaps):.1f}h")

with open(f"{BASE}/watch_warning_2019.json", "w") as f:
    json.dump({
        "watch_threshold": float(watch_thresh), "warning_threshold": float(warning_thresh),
        "n_true_onsets": n_true_onsets,
        "watch": {"n_detected": n_watch_detected, "mean_lead_hours": float(np.mean(watch_leads)),
                  "median_lead_hours": float(np.median(watch_leads)), "false_alarm_rate": float(watch_false)},
        "warning": {"n_detected": n_warning_detected, "mean_lead_hours": float(np.mean(warning_leads)),
                    "median_lead_hours": float(np.median(warning_leads)), "false_alarm_rate": float(warning_false)},
        "escalation_gap_mean_hours": float(np.mean(gaps)), "escalation_gap_median_hours": float(np.median(gaps)),
    }, f, indent=2)
print(f"\nsaved to {BASE}/watch_warning_2019.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: val=8689 test=8689

############### loading saved wind_graph 2019 model ###############

WARNING threshold (F1-optimal): 0.8469
WATCH threshold (precision>=0.25): 0.2949

==================== WATCH / WARNING TIERED SYSTEM, 2019 TEST (full 36h horizon) ====================
total true onsets: 7929

WATCH   (thresh=0.2949):  detected 6247/7929 (78.8%)  mean lead=6.79h  median lead=2.0h  false-alarm rate=8.4%
WARNING (thresh=0.8469):  detected 4600/7929 (58.0%)  mean lead=4.27h  median lead=0.0h  false-alarm rate=1.4%

ESCALATION GAP (extra hours WATCH buys over WARNING, for events caught by both, n=4600):
  mean=0.59h  median=0.0h

saved to /Users/drewbaldwin/PM2_5 Research/watch_warning_2019.json


In [2]:
# ================================================================
# CASE STUDY: pick 3 real test-set (window, region) examples where a
# sustained episode's onset falls early / middle / late within the
# first 24 hours of the model's own 36h output horizon. Run the saved
# wind_graph 2019 model (no retraining) and show its FULL predicted
# probability trajectory alongside the true PM2.5 concentration and
# true episode label, so we can see directly whether the model's
# confidence rises well before the onset or only right at it.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os, gc, json

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

norm = np.load(f"{BASE}/saved_wind_graph_2019/normalization.npz")
t_mean, t_std = norm["t_mean"], norm["t_std"]
s_mean, s_std = norm["s_mean"], norm["s_std"]
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != 2 or s_target != 2:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, ytrajl, sl = p_buckets[2]
    Xl.append(x_win)
    ytrajl.append(traj.T)
    sl.append(t)

X2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"test windows: {len(X2)}")

X2_t = torch.tensor(X2, dtype=torch.float32); del X2
starts2_t = torch.tensor(starts2, dtype=torch.long)
gc.collect()

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} loading saved wind_graph 2019 model {'#'*15}")
wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

test_probs = predict_traj_probs(wind_model, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# Find 3 clean examples: onset within the first 24h, in early/mid/late
# buckets, picked as the highest-peak (clearest) example per bucket.
# ================================================================
n_win = ytraj2.shape[0]
buckets = {"early (0-8h)": (0, 8), "middle (8-16h)": (8, 16), "late (16-24h)": (16, 24)}
candidates = {name: [] for name in buckets}

for w in range(n_win):
    for r in range(n_regions):
        lab24 = ytraj2[w, r, :24]
        if lab24.max() == 0:
            continue
        onset_h = int(np.argmax(lab24))
        for name, (lo, hi) in buckets.items():
            if lo <= onset_h < hi:
                t_start = starts2[w]
                peak_pm25 = region_pm25[t_start + WINDOW: t_start + WINDOW + 24, r].max()
                candidates[name].append((peak_pm25, w, r, onset_h))

chosen = {}
for name, cands in candidates.items():
    cands.sort(key=lambda x: -x[0])  # highest peak PM2.5 first = clearest example
    chosen[name] = cands[0]
    print(f"{name}: picked window={cands[0][1]} region={region_names[cands[0][2]]} "
          f"onset_hour={cands[0][3]}  peak_PM2.5={cands[0][0]:.1f}")

# ================================================================
# Print + plot each example's full 36h trajectory
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
PROB_COLOR = "#2E5AAC"
PM25_COLOR = "#B3541E"

for ax, (name, (peak, w, r, onset_h)) in zip(axes, chosen.items()):
    t_start = starts2[w]
    hours = np.arange(HORIZON)
    prob_traj = test_probs[w, r, :]
    label_traj = ytraj2[w, r, :]
    pm25_traj = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r]
    onset_dt = dt_index[t_start + WINDOW + onset_h]

    print(f"\n--- {name}: {region_names[r]}, forecast issued {dt_index[t_start + WINDOW - 1]}, "
          f"true onset at hour {onset_h} ({onset_dt}) ---")
    print("hour: " + " ".join(f"{h:5d}" for h in hours[:24]))
    print("prob: " + " ".join(f"{p:5.2f}" for p in prob_traj[:24]))
    print("PM25: " + " ".join(f"{p:5.0f}" for p in pm25_traj[:24]))
    print("label:" + " ".join(f"{int(l):5d}" for l in label_traj[:24]))

    ax2 = ax.twinx()
    ax.plot(hours, prob_traj, color=PROB_COLOR, linewidth=2, label="predicted P(episode)")
    ax.axhline(0.8469, color=PROB_COLOR, linestyle="--", linewidth=1, alpha=0.5, label="WARNING threshold")
    ax.axhline(0.2949, color=PROB_COLOR, linestyle=":", linewidth=1, alpha=0.5, label="WATCH threshold")
    ax2.plot(hours, pm25_traj, color=PM25_COLOR, linewidth=1.5, alpha=0.8, label="true PM2.5")
    ax2.axhline(75, color=PM25_COLOR, linestyle="--", linewidth=1, alpha=0.4)
    ax.axvline(onset_h, color="black", linewidth=1, alpha=0.6)
    ax.axvline(24, color="gray", linewidth=1, linestyle=":", alpha=0.5)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("P(episode)", color=PROB_COLOR)
    ax2.set_ylabel("PM2.5 (ug/m3)", color=PM25_COLOR)
    ax.set_title(f"{name}: {region_names[r]}, onset at hour {onset_h} ({onset_dt.strftime('%Y-%m-%d %H:%M')})")

axes[-1].set_xlabel("hours from forecast issue time")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{BASE}/fig_case_study_early_mid_late.png", dpi=200)
plt.close()
print(f"\nsaved figure to {BASE}/fig_case_study_early_mid_late.png")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
test windows: 8689

############### loading saved wind_graph 2019 model ###############
early (0-8h): picked window=1360 region=Jeonbuk onset_hour=7  peak_PM2.5=195.6
middle (8-16h): picked window=1352 region=Jeonbuk onset_hour=15  peak_PM2.5=195.6
late (16-24h): picked window=1351 region=Jeonbuk onset_hour=16  peak_PM2.5=178.1

--- early (0-8h): Jeonbuk, forecast issued 2019-02-28 03:00:00, true onset at hour 7 (2019-02-28 11:00:00) ---
hour:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23
prob:  0.92  0.91  0.91  0.91  0.91  0.91  0.92  0.91  0.91  0.90  0.91  0.91  0.89  0.91  0.91  0.89  0.89  0.90  0.89  0.87  0.88  0.87  0.86  0.86
PM25:    67    68    67    61    62    66    65    81    77    60    69    86   115   148   178   196   194   192   185   182   176   175   164   162
label:    0     0

In [3]:
# ================================================================
# PART A: how flat are the predicted probability trajectories,
# across the WHOLE test set? Compute within-window std and range
# (max-min) of test_probs[w,r,:] over the 36-hour horizon, for every
# (window, region), both overall and specifically for windows that
# contain a true episode (the policy-relevant subset).
# ================================================================
import numpy as np

within_window_std = test_probs.std(axis=-1)   # (n_win, n_regions)
within_window_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

has_episode = ytraj2.max(axis=-1) > 0  # (n_win, n_regions)

print(f"{'='*20} WITHIN-WINDOW PROBABILITY VARIABILITY, 2019 TEST {'='*20}")
print(f"\nALL (window, region) pairs (n={within_window_std.size}):")
print(f"  std:   mean={within_window_std.mean():.4f}  median={np.median(within_window_std):.4f}  "
      f"p90={np.percentile(within_window_std, 90):.4f}")
print(f"  range: mean={within_window_range.mean():.4f}  median={np.median(within_window_range):.4f}  "
      f"p90={np.percentile(within_window_range, 90):.4f}")

print(f"\nONLY windows containing a true episode (n={has_episode.sum()}):")
print(f"  std:   mean={within_window_std[has_episode].mean():.4f}  median={np.median(within_window_std[has_episode]):.4f}  "
      f"p90={np.percentile(within_window_std[has_episode], 90):.4f}")
print(f"  range: mean={within_window_range[has_episode].mean():.4f}  median={np.median(within_window_range[has_episode]):.4f}  "
      f"p90={np.percentile(within_window_range[has_episode], 90):.4f}")

# distribution of ranges, binned, for the episode subset -- how many are "basically flat" (range<0.1)?
ep_ranges = within_window_range[has_episode]
for cutoff in [0.05, 0.10, 0.20, 0.30, 0.50]:
    frac = (ep_ranges < cutoff).mean()
    print(f"  fraction of episode-windows with range < {cutoff}: {frac:.1%}")


==================== WITHIN-WINDOW PROBABILITY VARIABILITY, 2019 TEST ====================

ALL (window, region) pairs (n=147713):
  std:   mean=0.0203  median=0.0044  p90=0.0611
  range: mean=0.0680  median=0.0158  p90=0.2107

ONLY windows containing a true episode (n=7929):
  std:   mean=0.0705  median=0.0585  p90=0.1407
  range: mean=0.2326  median=0.2020  p90=0.4421
  fraction of episode-windows with range < 0.05: 6.6%
  fraction of episode-windows with range < 0.1: 17.6%
  fraction of episode-windows with range < 0.2: 49.4%
  fraction of episode-windows with range < 0.3: 73.2%
  fraction of episode-windows with range < 0.5: 93.4%


In [4]:
# ================================================================
# PART B: fixed example selection -- pick 3 genuinely independent
# episodes (deduplicated by absolute onset calendar time, not just
# window index), one each with onset in early/mid/late buckets of
# the first 24h. Then plot + print trajectories as before.
# ================================================================
import matplotlib.pyplot as plt

buckets = {"early (0-8h)": (0, 8), "middle (8-16h)": (8, 16), "late (16-24h)": (16, 24)}
candidates = {name: [] for name in buckets}

for w in range(ytraj2.shape[0]):
    for r in range(n_regions):
        lab24 = ytraj2[w, r, :24]
        if lab24.max() == 0:
            continue
        onset_h = int(np.argmax(lab24))
        t_start = starts2[w]
        onset_abs_hour = t_start + WINDOW + onset_h  # absolute index into dt_index
        for name, (lo, hi) in buckets.items():
            if lo <= onset_h < hi:
                peak_pm25 = region_pm25[t_start + WINDOW: t_start + WINDOW + 24, r].max()
                candidates[name].append((peak_pm25, w, r, onset_h, onset_abs_hour))

DEDUP_GAP_HOURS = 48  # minimum separation between chosen episodes' onset times
chosen = {}
used_onset_times = []  # (region, onset_abs_hour) of already-chosen examples

for name, cands in candidates.items():
    cands.sort(key=lambda x: -x[0])  # highest peak PM2.5 first
    for cand in cands:
        peak, w, r, onset_h, onset_abs_hour = cand
        conflict = any(r == used_r and abs(onset_abs_hour - used_t) < DEDUP_GAP_HOURS
                        for used_r, used_t in used_onset_times)
        if not conflict:
            chosen[name] = cand
            used_onset_times.append((r, onset_abs_hour))
            break
    else:
        print(f"WARNING: no independent candidate found for {name}")
        continue
    peak, w, r, onset_h, onset_abs_hour = chosen[name]
    print(f"{name}: window={w} region={region_names[r]} onset_hour={onset_h}  "
          f"peak_PM2.5={peak:.1f}  onset_time={dt_index[onset_abs_hour]}")

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
PROB_COLOR = "#2E5AAC"
PM25_COLOR = "#B3541E"

for ax, (name, (peak, w, r, onset_h, onset_abs_hour)) in zip(axes, chosen.items()):
    t_start = starts2[w]
    hours = np.arange(HORIZON)
    prob_traj = test_probs[w, r, :]
    label_traj = ytraj2[w, r, :]
    pm25_traj = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r]
    onset_dt = dt_index[onset_abs_hour]

    print(f"\n--- {name}: {region_names[r]}, forecast issued {dt_index[t_start + WINDOW - 1]}, "
          f"true onset at hour {onset_h} ({onset_dt}) ---")
    print("hour: " + " ".join(f"{h:5d}" for h in hours[:24]))
    print("prob: " + " ".join(f"{p:5.2f}" for p in prob_traj[:24]))
    print("PM25: " + " ".join(f"{p:5.0f}" for p in pm25_traj[:24]))
    print("label:" + " ".join(f"{int(l):5d}" for l in label_traj[:24]))

    ax2 = ax.twinx()
    ax.plot(hours, prob_traj, color=PROB_COLOR, linewidth=2, label="predicted P(episode)")
    ax.axhline(0.8469, color=PROB_COLOR, linestyle="--", linewidth=1, alpha=0.5, label="WARNING threshold")
    ax.axhline(0.2949, color=PROB_COLOR, linestyle=":", linewidth=1, alpha=0.5, label="WATCH threshold")
    ax2.plot(hours, pm25_traj, color=PM25_COLOR, linewidth=1.5, alpha=0.8, label="true PM2.5")
    ax2.axhline(75, color=PM25_COLOR, linestyle="--", linewidth=1, alpha=0.4)
    ax.axvline(onset_h, color="black", linewidth=1, alpha=0.6)
    ax.axvline(24, color="gray", linewidth=1, linestyle=":", alpha=0.5)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("P(episode)", color=PROB_COLOR)
    ax2.set_ylabel("PM2.5 (ug/m3)", color=PM25_COLOR)
    ax.set_title(f"{name}: {region_names[r]}, onset at hour {onset_h} ({onset_dt.strftime('%Y-%m-%d %H:%M')})")

axes[-1].set_xlabel("hours from forecast issue time")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{BASE}/fig_case_study_independent_episodes.png", dpi=200)
plt.close()
print(f"\nsaved figure to {BASE}/fig_case_study_independent_episodes.png")


early (0-8h): window=1360 region=Jeonbuk onset_hour=7  peak_PM2.5=195.6  onset_time=2019-02-28 11:00:00
middle (8-16h): window=1361 region=Sejong onset_hour=10  peak_PM2.5=191.0  onset_time=2019-02-28 15:00:00
late (16-24h): window=1354 region=Chungnam onset_hour=17  peak_PM2.5=155.0  onset_time=2019-02-28 15:00:00

--- early (0-8h): Jeonbuk, forecast issued 2019-02-28 03:00:00, true onset at hour 7 (2019-02-28 11:00:00) ---
hour:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23
prob:  0.92  0.91  0.91  0.91  0.91  0.91  0.92  0.91  0.91  0.90  0.91  0.91  0.89  0.91  0.91  0.89  0.89  0.90  0.89  0.87  0.88  0.87  0.86  0.86
PM25:    67    68    67    61    62    66    65    81    77    60    69    86   115   148   178   196   194   192   185   182   176   175   164   162
label:    0     0     0     0     0     0     0     1     0     0     0     1     1     1     1     1     1     1     1  

In [5]:
# ================================================================
# DIAGNOSTIC: are the flat probability trajectories a calibration
# artifact (sigmoid saturation compressing real logit differences)
# or a genuine architectural limitation (logits themselves are flat)?
# Recomputes raw LOGITS (pre-sigmoid) for the same test set, then
# compares logit spread vs probability spread, both for the 3 case-
# study examples and systematically across all episode windows.
# ================================================================
import numpy as np
import torch

def predict_traj_logits(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = logits.cpu().numpy()
    return out

# reload the saved model (predict_traj_probs already discarded it earlier)
wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

test_logits = predict_traj_logits(wind_model, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ---- systematic check: logit range vs probability range, episode windows ----
has_episode = ytraj2.max(axis=-1) > 0
logit_range = test_logits.max(axis=-1) - test_logits.min(axis=-1)
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"{'='*20} LOGIT vs PROBABILITY SPREAD, episode windows (n={has_episode.sum()}) {'='*20}")
print(f"logit range:  mean={logit_range[has_episode].mean():.3f}  median={np.median(logit_range[has_episode]):.3f}")
print(f"prob range:   mean={prob_range[has_episode].mean():.3f}  median={np.median(prob_range[has_episode]):.3f}")

# where in [0,1] do these trajectories typically sit? (mean probability level per window)
mean_prob_level = test_probs.mean(axis=-1)
print(f"\nmean probability level, episode windows: mean={mean_prob_level[has_episode].mean():.3f}  "
      f"median={np.median(mean_prob_level[has_episode]):.3f}")
# fraction of episode windows sitting mostly in the sigmoid's saturating zone (mean prob > 0.8 or < 0.2)
saturating = (mean_prob_level[has_episode] > 0.8) | (mean_prob_level[has_episode] < 0.2)
print(f"fraction of episode windows with mean prob in saturating zone (>0.8 or <0.2): {saturating.mean():.1%}")

# ---- the 3 case-study examples specifically ----
for name, (peak, w, r, onset_h, onset_abs_hour) in chosen.items():
    lg = test_logits[w, r, :24]
    pr = test_probs[w, r, :24]
    print(f"\n--- {name}: {region_names[r]} ---")
    print(f"logit range: {lg.max()-lg.min():.3f}  (min={lg.min():.2f} max={lg.max():.2f})")
    print(f"prob  range: {pr.max()-pr.min():.3f}  (min={pr.min():.3f} max={pr.max():.3f})")
    print("logits: " + " ".join(f"{v:5.2f}" for v in lg))


==================== LOGIT vs PROBABILITY SPREAD, episode windows (n=7929) ====================
logit range:  mean=2.452  median=2.543
prob range:   mean=0.233  median=0.202

mean probability level, episode windows: mean=0.642  median=0.815
fraction of episode windows with mean prob in saturating zone (>0.8 or <0.2): 71.6%

--- early (0-8h): Jeonbuk ---
logit range: 0.605  (min=1.82 max=2.43)
prob  range: 0.058  (min=0.861 max=0.919)
logits:  2.42  2.29  2.33  2.27  2.31  2.29  2.43  2.36  2.29  2.22  2.26  2.35  2.14  2.29  2.27  2.14  2.12  2.19  2.09  1.92  1.96  1.90  1.84  1.82

--- middle (8-16h): Sejong ---
logit range: 0.462  (min=0.82 max=1.28)
prob  range: 0.088  (min=0.695 max=0.783)
logits:  0.85  0.82  0.90  0.90  0.95  0.99  1.11  1.13  1.08  1.09  1.14  1.26  1.17  1.28  1.28  1.20  1.21  1.28  1.26  1.15  1.21  1.20  1.18  1.16

--- late (16-24h): Chungnam ---
logit range: 1.749  (min=-2.90 max=-1.15)
prob  range: 0.188  (min=0.052 max=0.240)
logits: -2.90 -2.65 -2.53 -

In [6]:
# ================================================================
# TEMPERATURE SCALING: fit a single scalar T on VALIDATION logits
# (recalibrated_prob = sigmoid(logit / T)), then re-evaluate flatness,
# thresholds, and lead-time/detection on the TEST set using the
# recalibrated probabilities. AUC-PR/AUC-ROC should be unchanged
# (monotonic transform preserves ranking) -- included as a sanity check.
# ================================================================
import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

def predict_traj_logits(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = logits.cpu().numpy()
    return out

wind_encoder = WindConvLayer(n_feats, 32).to(DEVICE)
wind_model = RegionFromTrajectoryGCN(wind_encoder, region_membership_t, dropout=0.5).to(DEVICE)
wind_model.load_state_dict(torch.load(f"{BASE}/saved_wind_graph_2019/phase2_model.pt", map_location=DEVICE))
wind_model.eval()

val_logits = predict_traj_logits(wind_model, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
del wind_model; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# Fit temperature T on validation (grid search -- simple, robust for
# a single scalar): minimize BCE(val_logits/T, val_labels)
# ================================================================
val_logits_flat = val_logits.reshape(-1)
val_labels_flat = ytraj1.reshape(-1)

def bce_for_T(T):
    p = 1 / (1 + np.exp(-val_logits_flat / T))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(val_labels_flat * np.log(p) + (1 - val_labels_flat) * np.log(1 - p))

T_grid = np.arange(0.5, 5.01, 0.05)
bce_vals = [bce_for_T(T) for T in T_grid]
best_T = T_grid[np.argmin(bce_vals)]
print(f"best temperature T = {best_T:.2f}  (BCE={min(bce_vals):.4f}, vs T=1.0 BCE={bce_for_T(1.0):.4f})")

recal_val_probs = (1 / (1 + np.exp(-val_logits / best_T))).astype(np.float32)
recal_test_probs = (1 / (1 + np.exp(-test_logits / best_T))).astype(np.float32)

# ---- sanity check: AUC-PR/AUC-ROC should be unchanged (monotonic transform) ----
raw_auc_pr = average_precision_score(ytraj2.max(axis=-1).reshape(-1), test_probs.max(axis=-1).reshape(-1))
recal_auc_pr = average_precision_score(ytraj2.max(axis=-1).reshape(-1), recal_test_probs.max(axis=-1).reshape(-1))
raw_auc_roc = roc_auc_score(ytraj2.max(axis=-1).reshape(-1), test_probs.max(axis=-1).reshape(-1))
recal_auc_roc = roc_auc_score(ytraj2.max(axis=-1).reshape(-1), recal_test_probs.max(axis=-1).reshape(-1))
print(f"\nsanity check -- AUC-PR: raw={raw_auc_pr:.4f}  recalibrated={recal_auc_pr:.4f}  (should match)")
print(f"sanity check -- AUC-ROC: raw={raw_auc_roc:.4f}  recalibrated={recal_auc_roc:.4f}  (should match)")

# ================================================================
# Re-check flatness on recalibrated probabilities
# ================================================================
has_episode = ytraj2.max(axis=-1) > 0
recal_range = recal_test_probs.max(axis=-1) - recal_test_probs.min(axis=-1)
raw_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} FLATNESS: RAW vs RECALIBRATED, episode windows (n={has_episode.sum()}) {'='*20}")
print(f"raw prob range:          mean={raw_range[has_episode].mean():.4f}  median={np.median(raw_range[has_episode]):.4f}")
print(f"recalibrated prob range: mean={recal_range[has_episode].mean():.4f}  median={np.median(recal_range[has_episode]):.4f}")

print(f"\nthe 3 case-study examples:")
for name, (peak, w, r, onset_h, onset_abs_hour) in chosen.items():
    rr = recal_test_probs[w, r, :24]
    print(f"  {name} ({region_names[r]}): raw range={test_probs[w,r,:24].max()-test_probs[w,r,:24].min():.3f}  "
          f"recalibrated range={rr.max()-rr.min():.3f}  (recalibrated: {' '.join(f'{v:.2f}' for v in rr)})")

# ================================================================
# Re-select WATCH/WARNING thresholds on RECALIBRATED validation probs,
# then re-run the lead-time/detection/false-alarm analysis on TEST
# ================================================================
val_score = recal_val_probs.max(axis=-1).reshape(-1)
val_label = ytraj1.max(axis=-1).reshape(-1)

fpr, tpr, thresholds = roc_curve(val_label, val_score)
f1s, precisions = [], []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(val_label, pred, zero_division=0)
    r = recall_score(val_label, pred, zero_division=0)
    precisions.append(p)
    f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
precisions = np.array(precisions)

warning_thresh = thresholds[np.argmax(f1s)]
PRECISION_FLOOR = 0.25
eligible = np.where(precisions >= PRECISION_FLOOR)[0]
watch_thresh = thresholds[eligible[np.argmin(thresholds[eligible])]] if len(eligible) > 0 else warning_thresh
print(f"\nrecalibrated WARNING threshold: {warning_thresh:.4f}  (was 0.8469 raw)")
print(f"recalibrated WATCH threshold:   {watch_thresh:.4f}  (was 0.2949 raw)")

n_win, n_reg, H = ytraj2.shape
watch_leads, warning_leads = [], []
n_true_onsets = 0
n_watch_detected = 0
n_warning_detected = 0

for w in range(n_win):
    for r in range(n_reg):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        n_true_onsets += 1
        onset_h = int(np.argmax(lab))
        prob = recal_test_probs[w, r, :]

        watch_idx = np.where(prob >= watch_thresh)[0]
        watch_idx = watch_idx[watch_idx <= onset_h]
        warning_idx = np.where(prob >= warning_thresh)[0]
        warning_idx = warning_idx[warning_idx <= onset_h]

        if len(watch_idx) > 0:
            n_watch_detected += 1
            watch_leads.append(onset_h - watch_idx[0])
        if len(warning_idx) > 0:
            n_warning_detected += 1
            warning_leads.append(onset_h - warning_idx[0])

non_onset_mask = ytraj2.max(axis=-1) == 0
watch_false = np.mean((recal_test_probs >= watch_thresh).any(axis=-1)[non_onset_mask])
warning_false = np.mean((recal_test_probs >= warning_thresh).any(axis=-1)[non_onset_mask])

print(f"\n{'='*20} WATCH / WARNING, RECALIBRATED PROBABILITIES {'='*20}")
print(f"total true onsets: {n_true_onsets}")
print(f"WATCH   (thresh={watch_thresh:.4f}):  detected {n_watch_detected}/{n_true_onsets} ({n_watch_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(watch_leads):.2f}h  median lead={np.median(watch_leads):.1f}h  false-alarm rate={watch_false:.1%}")
print(f"WARNING (thresh={warning_thresh:.4f}):  detected {n_warning_detected}/{n_true_onsets} ({n_warning_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(warning_leads):.2f}h  median lead={np.median(warning_leads):.1f}h  false-alarm rate={warning_false:.1%}")

print(f"\n{'='*20} COMPARE TO RAW (pre-recalibration) {'='*20}")
print(f"RAW WATCH:   detected 78.8%  mean lead=6.79h  median lead=2.0h  false-alarm=8.4%")
print(f"RAW WARNING: detected 58.0%  mean lead=4.27h  median lead=0.0h  false-alarm=1.4%")


best temperature T = 1.10  (BCE=0.1379, vs T=1.0 BCE=0.1389)

sanity check -- AUC-PR: raw=0.7154  recalibrated=0.7154  (should match)
sanity check -- AUC-ROC: raw=0.9580  recalibrated=0.9580  (should match)

==================== FLATNESS: RAW vs RECALIBRATED, episode windows (n=7929) ====================
raw prob range:          mean=0.2326  median=0.2020
recalibrated prob range: mean=0.2339  median=0.2117

the 3 case-study examples:
  early (0-8h) (Jeonbuk): raw range=0.058  recalibrated range=0.061  (recalibrated: 0.90 0.89 0.89 0.89 0.89 0.89 0.90 0.90 0.89 0.88 0.89 0.89 0.88 0.89 0.89 0.87 0.87 0.88 0.87 0.85 0.86 0.85 0.84 0.84)
  middle (8-16h) (Sejong): raw range=0.088  recalibrated range=0.084  (recalibrated: 0.68 0.68 0.69 0.69 0.70 0.71 0.73 0.74 0.73 0.73 0.74 0.76 0.74 0.76 0.76 0.75 0.75 0.76 0.76 0.74 0.75 0.75 0.75 0.74)
  late (16-24h) (Chungnam): raw range=0.188  recalibrated range=0.193  (recalibrated: 0.07 0.08 0.09 0.10 0.10 0.10 0.11 0.13 0.13 0.14 0.14 0.15 0.16 

In [7]:
# ================================================================
# ISOTONIC REGRESSION CALIBRATION: more flexible than temperature
# scaling (not constrained to a single global scalar). Fit on
# VALIDATION (probability -> empirical outcome), apply to TEST,
# re-check flatness, AUC-PR/ROC sanity, thresholds, and lead-time.
# ================================================================
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, roc_curve

val_probs = 1 / (1 + np.exp(-val_logits))  # derive from logits already in memory

val_probs_flat = val_probs.reshape(-1)
val_labels_flat = ytraj1.reshape(-1)

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(val_probs_flat, val_labels_flat)

iso_val_probs = iso.predict(val_probs.reshape(-1)).reshape(val_probs.shape).astype(np.float32)
iso_test_probs = iso.predict(test_probs.reshape(-1)).reshape(test_probs.shape).astype(np.float32)

# ---- sanity check: AUC-PR/AUC-ROC should be unchanged (monotonic transform) ----
raw_auc_pr = average_precision_score(ytraj2.max(axis=-1).reshape(-1), test_probs.max(axis=-1).reshape(-1))
iso_auc_pr = average_precision_score(ytraj2.max(axis=-1).reshape(-1), iso_test_probs.max(axis=-1).reshape(-1))
raw_auc_roc = roc_auc_score(ytraj2.max(axis=-1).reshape(-1), test_probs.max(axis=-1).reshape(-1))
iso_auc_roc = roc_auc_score(ytraj2.max(axis=-1).reshape(-1), iso_test_probs.max(axis=-1).reshape(-1))
print(f"sanity check -- AUC-PR: raw={raw_auc_pr:.4f}  isotonic={iso_auc_pr:.4f}  (should be very close)")
print(f"sanity check -- AUC-ROC: raw={raw_auc_roc:.4f}  isotonic={iso_auc_roc:.4f}  (should be very close)")
print(f"(note: isotonic can create small plateaus/ties that AUC-PR/ROC are sensitive to, unlike temperature scaling's strictly monotonic rescale)")

# ================================================================
# Re-check flatness
# ================================================================
has_episode = ytraj2.max(axis=-1) > 0
iso_range = iso_test_probs.max(axis=-1) - iso_test_probs.min(axis=-1)
raw_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} FLATNESS: RAW vs ISOTONIC, episode windows (n={has_episode.sum()}) {'='*20}")
print(f"raw prob range:      mean={raw_range[has_episode].mean():.4f}  median={np.median(raw_range[has_episode]):.4f}")
print(f"isotonic prob range: mean={iso_range[has_episode].mean():.4f}  median={np.median(iso_range[has_episode]):.4f}")

print(f"\nthe 3 case-study examples:")
for name, (peak, w, r, onset_h, onset_abs_hour) in chosen.items():
    rr = iso_test_probs[w, r, :24]
    print(f"  {name} ({region_names[r]}): raw range={test_probs[w,r,:24].max()-test_probs[w,r,:24].min():.3f}  "
          f"isotonic range={rr.max()-rr.min():.3f}  (isotonic: {' '.join(f'{v:.2f}' for v in rr)})")

# ================================================================
# Re-select WATCH/WARNING thresholds on ISOTONIC validation probs,
# then re-run lead-time/detection/false-alarm analysis on TEST
# ================================================================
val_score = iso_val_probs.max(axis=-1).reshape(-1)
val_label = ytraj1.max(axis=-1).reshape(-1)

fpr, tpr, thresholds = roc_curve(val_label, val_score)
f1s, precisions = [], []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(val_label, pred, zero_division=0)
    r = recall_score(val_label, pred, zero_division=0)
    precisions.append(p)
    f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
precisions = np.array(precisions)

warning_thresh = thresholds[np.argmax(f1s)]
PRECISION_FLOOR = 0.25
eligible = np.where(precisions >= PRECISION_FLOOR)[0]
watch_thresh = thresholds[eligible[np.argmin(thresholds[eligible])]] if len(eligible) > 0 else warning_thresh
print(f"\nisotonic WARNING threshold: {warning_thresh:.4f}  (was 0.8469 raw)")
print(f"isotonic WATCH threshold:   {watch_thresh:.4f}  (was 0.2949 raw)")

n_win, n_reg, H = ytraj2.shape
watch_leads, warning_leads = [], []
n_true_onsets = 0
n_watch_detected = 0
n_warning_detected = 0

for w in range(n_win):
    for r in range(n_reg):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        n_true_onsets += 1
        onset_h = int(np.argmax(lab))
        prob = iso_test_probs[w, r, :]

        watch_idx = np.where(prob >= watch_thresh)[0]
        watch_idx = watch_idx[watch_idx <= onset_h]
        warning_idx = np.where(prob >= warning_thresh)[0]
        warning_idx = warning_idx[warning_idx <= onset_h]

        if len(watch_idx) > 0:
            n_watch_detected += 1
            watch_leads.append(onset_h - watch_idx[0])
        if len(warning_idx) > 0:
            n_warning_detected += 1
            warning_leads.append(onset_h - warning_idx[0])

non_onset_mask = ytraj2.max(axis=-1) == 0
watch_false = np.mean((iso_test_probs >= watch_thresh).any(axis=-1)[non_onset_mask])
warning_false = np.mean((iso_test_probs >= warning_thresh).any(axis=-1)[non_onset_mask])

print(f"\n{'='*20} WATCH / WARNING, ISOTONIC-CALIBRATED PROBABILITIES {'='*20}")
print(f"total true onsets: {n_true_onsets}")
print(f"WATCH   (thresh={watch_thresh:.4f}):  detected {n_watch_detected}/{n_true_onsets} ({n_watch_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(watch_leads):.2f}h  median lead={np.median(watch_leads):.1f}h  false-alarm rate={watch_false:.1%}")
print(f"WARNING (thresh={warning_thresh:.4f}):  detected {n_warning_detected}/{n_true_onsets} ({n_warning_detected/n_true_onsets:.1%})  "
      f"mean lead={np.mean(warning_leads):.2f}h  median lead={np.median(warning_leads):.1f}h  false-alarm rate={warning_false:.1%}")

print(f"\n{'='*20} COMPARE TO RAW (pre-calibration) {'='*20}")
print(f"RAW WATCH:   detected 78.8%  mean lead=6.79h  median lead=2.0h  false-alarm=8.4%")
print(f"RAW WARNING: detected 58.0%  mean lead=4.27h  median lead=0.0h  false-alarm=1.4%")


sanity check -- AUC-PR: raw=0.7154  isotonic=0.7094  (should be very close)
sanity check -- AUC-ROC: raw=0.9580  isotonic=0.9579  (should be very close)
(note: isotonic can create small plateaus/ties that AUC-PR/ROC are sensitive to, unlike temperature scaling's strictly monotonic rescale)

==================== FLATNESS: RAW vs ISOTONIC, episode windows (n=7929) ====================
raw prob range:      mean=0.2326  median=0.2020
isotonic prob range: mean=0.2744  median=0.0737

the 3 case-study examples:
  early (0-8h) (Jeonbuk): raw range=0.058  isotonic range=0.022  (isotonic: 0.11 0.10 0.10 0.10 0.10 0.10 0.11 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.10 0.09 0.09 0.09 0.08 0.08)
  middle (8-16h) (Sejong): raw range=0.088  isotonic range=0.011  (isotonic: 0.04 0.04 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05)
  late (16-24h) (Chungnam): raw range=0.188  isotonic range=0.013  (isotonic: 0.01 0.01 0.01 0.

In [1]:
# ================================================================
# DECODER ARCHITECTURE: replaces the single Linear(gru_hidden, HORIZON)
# readout with a genuine per-timestep GRU decoder that unrolls across
# the 24 future hours. The encoder (reading the 36-hour input window)
# is unchanged; its final hidden state now seeds a SECOND GRU that
# processes 24 learned positional-step embeddings, producing a
# DISTINCT hidden state (and thus a distinct logit) for each future
# hour, instead of 24 linear combinations of one fixed vector.
# HORIZON=24 (future), WINDOW=36 (input, unchanged).
# Split: train=2016-2017/val=2018/test=2019, single seed=0 first look.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24  # <-- future horizon, now decoded per-timestep instead of 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN_Decoder(nn.Module):
    """Same encoder side as the headline model (station_conv -> attn_pool ->
    region_conv -> region_gru over the 36 INPUT hours), but the output side
    is now a genuine per-timestep GRU decoder: the encoder's final hidden
    state seeds a second GRU that unrolls across HORIZON learned positional
    step-embeddings, giving each future hour its OWN hidden state (and thus
    its own logit) instead of 24 linear combinations of one fixed vector."""
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)  # encoder, over input window
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)  # decoder, over output horizon
        self.output_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local
        self.horizon = horizon

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)  # (1, B*n_reg, gru_hidden) -- encoder summary

        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)  # (B*n_reg, horizon, decoder_dim)
        dec_out, _ = self.decoder_gru(pos, h_final)  # (B*n_reg, horizon, gru_hidden)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)  # (B*n_reg, horizon)
        return logits.reshape(B, n_reg, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} PHASE 1: station quantile pretraining (seed=0) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
p1 = StationQuantileGCN(n_feats, dropout=0.5).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=0.003, weight_decay=0.0005)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}")
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
del p1; gc.collect()

print(f"\n{'#'*15} PHASE 2: decoder-based region classifier (seed=0) {'#'*15}")
p2 = RegionFromTrajectoryGCN_Decoder(encoder_copy, region_membership_t, dropout=0.5).to(DEVICE)
opt2 = torch.optim.Adam(p2.parameters(), lr=0.003, weight_decay=0.0005)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    tl = run_p2_traj_epoch(p2, X0_t, ytraj0_t, starts0_t, opt2, True)
    vl = run_p2_traj_epoch(p2, X1_t, ytraj1_t, starts1_t, opt2, False)
    val_probs = predict_traj_probs(p2, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}  val_aucpr={va:.4f}")
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
p2.load_state_dict(best_state)
p2.eval()

# ---- evaluate on test ----
test_probs = predict_traj_probs(p2, X2_t, starts2_t)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)
test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

print(f"\n{'='*20} DECODER MODEL RESULTS, 2019 TEST (HORIZON={HORIZON}h) {'='*20}")
print(f"val_AUCPR={best_val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")

# ---- flatness check: did the decoder actually fix the differentiation issue? ----
has_episode = ytraj2.max(axis=-1) > 0
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)
print(f"\n{'='*20} FLATNESS CHECK (compare to old single-Linear-head model) {'='*20}")
print(f"episode windows (n={has_episode.sum()}):")
print(f"  prob range: mean={prob_range[has_episode].mean():.4f}  median={np.median(prob_range[has_episode]):.4f}")
print(f"  (old single-Linear-head model had: mean=0.2326  median=0.2020)")

with open(f"{BASE}/decoder_model_2019_seed0.json", "w") as f:
    json.dump({"val_aucpr": float(best_val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "prob_range_mean": float(prob_range[has_episode].mean()), "prob_range_median": float(np.median(prob_range[has_episode]))}, f, indent=2)
print(f"\nsaved to {BASE}/decoder_model_2019_seed0.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

############### PHASE 1: station quantile pretraining (seed=0) ###############
  epoch 1  train_loss=0.1881  val_loss=0.1730
  epoch 2  train_loss=0.1808  val_loss=0.1747

############### PHASE 2: decoder-based region classifier (seed=0) ###############
  epoch 1  train_loss=0.3773  val_loss=0.2913  val_aucpr=0.5323
  epoch 2  train_loss=0.2690  val_loss=0.2933  val_aucpr=0.5506

==================== DECODER MODEL RESULTS, 2019 TEST (HORIZON=24h) ====================
val_AUCPR=0.5506  test_AUCROC=0.9775  test_AUCPR=0.7757

==================== FLATNESS CHECK (compare to old single-Linear-head model) ====================
episode windows (n=6426):
  prob range: mean=0.1668  median=0.0497
  (old single-Linear-head model had: mean=0.2326  median=0.2020)

saved to /Users/drewbaldwin/PM2_5 Research/decoder_model_2019_seed0.json


In [1]:
# ================================================================
# DECODER MODEL v2: (a) longer phase-2 training with patience-based
# early stopping and per-epoch flatness tracking, to see if the
# decoder just needed more time; (b) an auxiliary LOCALIZATION loss
# -- cross-entropy over the 24 logits against the true onset hour,
# for windows that contain an episode -- which directly forces the
# model to concentrate probability mass at the correct hour RELATIVE
# to the others in the same window, unlike independent per-hour BCE.
#
# Set LOCALIZATION_WEIGHT = 0.0 to test "just train longer" (step 1).
# Set LOCALIZATION_WEIGHT > 0 to test the new loss (step 2).
# Run it both ways and compare.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"

# ---- CONFIG: change this and rerun to compare the two experiments ----
LOCALIZATION_WEIGHT = 0.0   # 0.0 = step 1 (longer training only); try e.g. 0.3 for step 2
MAX_EPOCHS_P2 = 15
PATIENCE = 5
RUN_TAG = "longer_training" if LOCALIZATION_WEIGHT == 0.0 else f"localization_w{LOCALIZATION_WEIGHT}"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN_Decoder(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local
        self.horizon = horizon

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)

        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_reg, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"device={DEVICE}  RUN_TAG={RUN_TAG}  LOCALIZATION_WEIGHT={LOCALIZATION_WEIGHT}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_bce, total_loc, total_n = 0.0, 0.0, 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                bce = region_criterion(logits, yb)

                loc_loss = torch.tensor(0.0, device=DEVICE)
                if LOCALIZATION_WEIGHT > 0:
                    # flatten (B, n_reg) -> episode-level: target = argmax hour, only for
                    # (window, region) pairs that actually have an episode in this horizon
                    B_, R_, H_ = logits.shape
                    logits_flat = logits.reshape(B_ * R_, H_)
                    yb_flat = yb.reshape(B_ * R_, H_)
                    has_ep = yb_flat.max(dim=-1).values > 0
                    if has_ep.any():
                        target_hour = yb_flat[has_ep].argmax(dim=-1)
                        loc_loss = F.cross_entropy(logits_flat[has_ep], target_hour)

                loss = bce + LOCALIZATION_WEIGHT * loc_loss
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_bce += bce.item() * len(mb_idx)
            total_loc += float(loc_loss) * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1), total_bce / max(total_n, 1), total_loc / max(total_n, 1)

def predict_traj_probs(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} PHASE 1: station quantile pretraining (seed=0) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
p1 = StationQuantileGCN(n_feats, dropout=0.5).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=0.003, weight_decay=0.0005)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}")
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
del p1; gc.collect()

print(f"\n{'#'*15} PHASE 2: decoder classifier, {RUN_TAG} (seed=0) {'#'*15}")
p2 = RegionFromTrajectoryGCN_Decoder(encoder_copy, region_membership_t, dropout=0.5).to(DEVICE)
opt2 = torch.optim.Adam(p2.parameters(), lr=0.003, weight_decay=0.0005)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
has_episode_val = ytraj1.max(axis=-1) > 0

best_val_aucpr, best_state, best_epoch = -1.0, None, 0
patience_counter = 0
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    tl, tbce, tloc = run_p2_traj_epoch(p2, X0_t, ytraj0_t, starts0_t, opt2, True)
    vl, vbce, vloc = run_p2_traj_epoch(p2, X1_t, ytraj1_t, starts1_t, opt2, False)
    val_probs = predict_traj_probs(p2, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    val_range = val_probs.max(axis=-1) - val_probs.min(axis=-1)
    val_flatness = float(np.median(val_range[has_episode_val]))
    print(f"  epoch {epoch}  train_loss={tl:.4f} (bce={tbce:.4f} loc={tloc:.4f})  "
          f"val_loss={vl:.4f}  val_aucpr={va:.4f}  val_median_range={val_flatness:.4f}")
    if va > best_val_aucpr:
        best_val_aucpr, best_state, best_epoch = va, copy.deepcopy(p2.state_dict()), epoch
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (best was epoch {best_epoch})")
            break

p2.load_state_dict(best_state)
p2.eval()

test_probs = predict_traj_probs(p2, X2_t, starts2_t)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)
test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

has_episode = ytraj2.max(axis=-1) > 0
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} RESULTS: {RUN_TAG}, 2019 TEST (HORIZON={HORIZON}h, best_epoch={best_epoch}) {'='*20}")
print(f"val_AUCPR={best_val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
print(f"episode windows (n={has_episode.sum()}): prob range mean={prob_range[has_episode].mean():.4f}  median={np.median(prob_range[has_episode]):.4f}")
print(f"  (2-epoch decoder had: mean=0.1668 median=0.0497 | old single-Linear-head had: mean=0.2326 median=0.2020)")

with open(f"{BASE}/decoder_model_2019_{RUN_TAG}.json", "w") as f:
    json.dump({"run_tag": RUN_TAG, "localization_weight": LOCALIZATION_WEIGHT, "best_epoch": best_epoch,
               "val_aucpr": float(best_val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "prob_range_mean": float(prob_range[has_episode].mean()), "prob_range_median": float(np.median(prob_range[has_episode]))}, f, indent=2)
print(f"\nsaved to {BASE}/decoder_model_2019_{RUN_TAG}.json")


device=mps  RUN_TAG=longer_training  LOCALIZATION_WEIGHT=0.0
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

############### PHASE 1: station quantile pretraining (seed=0) ###############
  epoch 1  train_loss=0.1881  val_loss=0.1731
  epoch 2  train_loss=0.1808  val_loss=0.1747

############### PHASE 2: decoder classifier, longer_training (seed=0) ###############
  epoch 1  train_loss=0.3773 (bce=0.3773 loc=0.0000)  val_loss=0.2908  val_aucpr=0.5315  val_median_range=0.0395
  epoch 2  train_loss=0.2688 (bce=0.2688 loc=0.0000)  val_loss=0.2847  val_aucpr=0.5453  val_median_range=0.1575
  epoch 3  train_loss=0.2441 (bce=0.2441 loc=0.0000)  val_loss=0.2952  val_aucpr=0.5368  val_median_range=0.2352
  epoch 4  train_loss=0.2194 (bce=0.2194 loc=0.0000)  val_loss=0.2960  val_aucpr=0.5349  val_median_range=0.1970
  epoch 5  train_loss=0.2026 (bce=0.2026 loc=0.0000)  val_loss=0.3239  val_aucpr=0.5009  v

In [1]:
# ================================================================
# LOSS-FUNCTION APPROACH: decoder model trained with a combined loss
# that directly targets BOTH stated objectives:
#   (1) CLASSIFY whether an extreme event occurs in the next 24h
#       window -- standard per-hour BCE loss (same as always).
#   (2) LOCALIZE where in that window it occurs -- a new auxiliary
#       cross-entropy term: for every (window, region) that DOES
#       contain an episode, treat the 24 logits as an unnormalized
#       distribution over "which hour is the onset" and train against
#       the true onset hour with softmax cross-entropy. Unlike BCE
#       (which scores each hour independently), cross-entropy is
#       normalized across the 24 hours -- raising one hour's score
#       necessarily lowers the others -- which is exactly the pressure
#       needed to stop the model from just being uniformly flat.
# Total loss = BCE + LOCALIZATION_WEIGHT * localization_CE.
# Same decoder architecture as before (station/region wind-graph
# encoder -> GRU decoder unrolling 24 future hours).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
LOCALIZATION_WEIGHT = 0.3
MAX_EPOCHS_P2 = 15
PATIENCE = 5
RUN_TAG = f"localization_w{LOCALIZATION_WEIGHT}"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN_Decoder(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local
        self.horizon = horizon

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)

        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_reg, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"device={DEVICE}  RUN_TAG={RUN_TAG}  LOCALIZATION_WEIGHT={LOCALIZATION_WEIGHT}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_bce, total_loc, total_n = 0.0, 0.0, 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                # ---- objective 1: classify if an event occurs (per-hour BCE) ----
                bce = region_criterion(logits, yb)

                # ---- objective 2: localize WHEN it occurs (cross-entropy over
                # hours, only for window/region pairs that DO contain an episode) ----
                B_, R_, H_ = logits.shape
                logits_flat = logits.reshape(B_ * R_, H_)
                yb_flat = yb.reshape(B_ * R_, H_)
                has_ep = yb_flat.max(dim=-1).values > 0
                if has_ep.any():
                    target_hour = yb_flat[has_ep].argmax(dim=-1)
                    loc_loss = F.cross_entropy(logits_flat[has_ep], target_hour)
                else:
                    loc_loss = torch.tensor(0.0, device=DEVICE)

                loss = bce + LOCALIZATION_WEIGHT * loc_loss
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_bce += bce.item() * len(mb_idx)
            total_loc += float(loc_loss) * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1), total_bce / max(total_n, 1), total_loc / max(total_n, 1)

def predict_traj_probs(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} PHASE 1: station quantile pretraining (seed=0) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
p1 = StationQuantileGCN(n_feats, dropout=0.5).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=0.003, weight_decay=0.0005)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}")
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
del p1; gc.collect()

print(f"\n{'#'*15} PHASE 2: decoder classifier + localization loss (seed=0) {'#'*15}")
p2 = RegionFromTrajectoryGCN_Decoder(encoder_copy, region_membership_t, dropout=0.5).to(DEVICE)
opt2 = torch.optim.Adam(p2.parameters(), lr=0.003, weight_decay=0.0005)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
has_episode_val = ytraj1.max(axis=-1) > 0

best_val_aucpr, best_state, best_epoch = -1.0, None, 0
patience_counter = 0
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    tl, tbce, tloc = run_p2_traj_epoch(p2, X0_t, ytraj0_t, starts0_t, opt2, True)
    vl, vbce, vloc = run_p2_traj_epoch(p2, X1_t, ytraj1_t, starts1_t, opt2, False)
    val_probs = predict_traj_probs(p2, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    val_range = val_probs.max(axis=-1) - val_probs.min(axis=-1)
    val_flatness = float(np.median(val_range[has_episode_val]))
    print(f"  epoch {epoch}  train_loss={tl:.4f} (bce={tbce:.4f} loc={tloc:.4f})  "
          f"val_loss={vl:.4f} (bce={vbce:.4f} loc={vloc:.4f})  val_aucpr={va:.4f}  val_median_range={val_flatness:.4f}")
    if va > best_val_aucpr:
        best_val_aucpr, best_state, best_epoch = va, copy.deepcopy(p2.state_dict()), epoch
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (best was epoch {best_epoch})")
            break

p2.load_state_dict(best_state)
p2.eval()

test_probs = predict_traj_probs(p2, X2_t, starts2_t)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)
test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

has_episode = ytraj2.max(axis=-1) > 0
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} RESULTS: {RUN_TAG}, 2019 TEST (HORIZON={HORIZON}h, best_epoch={best_epoch}) {'='*20}")
print(f"val_AUCPR={best_val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
print(f"episode windows (n={has_episode.sum()}): prob range mean={prob_range[has_episode].mean():.4f}  median={np.median(prob_range[has_episode]):.4f}")
print(f"  (2-epoch decoder, no localization loss, had: mean=0.1668 median=0.0497)")
print(f"  (original single-Linear-head model had: mean=0.2326 median=0.2020)")

with open(f"{BASE}/decoder_model_2019_{RUN_TAG}.json", "w") as f:
    json.dump({"run_tag": RUN_TAG, "localization_weight": LOCALIZATION_WEIGHT, "best_epoch": best_epoch,
               "val_aucpr": float(best_val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "prob_range_mean": float(prob_range[has_episode].mean()), "prob_range_median": float(np.median(prob_range[has_episode]))}, f, indent=2)
print(f"\nsaved to {BASE}/decoder_model_2019_{RUN_TAG}.json")


device=mps  RUN_TAG=localization_w0.3  LOCALIZATION_WEIGHT=0.3
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

############### PHASE 1: station quantile pretraining (seed=0) ###############
  epoch 1  train_loss=0.1881  val_loss=0.1730
  epoch 2  train_loss=0.1808  val_loss=0.1747

############### PHASE 2: decoder classifier + localization loss (seed=0) ###############


/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_3867/2551619127.py:424: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  total_loc += float(loc_loss) * len(mb_idx); total_n += len(mb_idx)


  epoch 1  train_loss=1.1575 (bce=0.3932 loc=2.5479)  val_loss=0.4168 (bce=0.2930 loc=0.4126)  val_aucpr=0.5317  val_median_range=0.1854
  epoch 2  train_loss=1.0134 (bce=0.2744 loc=2.4632)  val_loss=0.4150 (bce=0.2935 loc=0.4050)  val_aucpr=0.5439  val_median_range=0.1702
  epoch 3  train_loss=0.9703 (bce=0.2472 loc=2.4103)  val_loss=0.4291 (bce=0.3099 loc=0.3976)  val_aucpr=0.5267  val_median_range=0.2387
  epoch 4  train_loss=0.9336 (bce=0.2261 loc=2.3586)  val_loss=0.4365 (bce=0.3128 loc=0.4124)  val_aucpr=0.5296  val_median_range=0.2839
  epoch 5  train_loss=0.8963 (bce=0.2099 loc=2.2881)  val_loss=0.5271 (bce=0.4015 loc=0.4184)  val_aucpr=0.4861  val_median_range=0.2781
  epoch 6  train_loss=0.8946 (bce=0.2022 loc=2.3082)  val_loss=0.4757 (bce=0.3529 loc=0.4094)  val_aucpr=0.5256  val_median_range=0.3421
  epoch 7  train_loss=0.8471 (bce=0.1880 loc=2.1972)  val_loss=0.4854 (bce=0.3616 loc=0.4127)  val_aucpr=0.5077  val_median_range=0.2908
  early stopping at epoch 7 (best was epo

In [2]:
# ================================================================
# CASE STUDY v2: pick 3 GENUINELY DISTINCT 24-hour episodes (onset
# times separated by at least 7 days, not just different regions of
# the same national event -- the mistake from last time), one each
# with onset early / middle / late in the 24h window. Run the TRAINED
# decoder+localization model (p2, already in memory from the previous
# cell) and compare predicted probability trajectories against the
# true PM2.5 and true label trajectories.
# ================================================================
import numpy as np
import matplotlib.pyplot as plt

DEDUP_GAP_HOURS = 24 * 7  # require onset times at least 7 days apart

buckets = {"early (0-8h)": (0, 8), "middle (8-16h)": (8, 16), "late (16-24h)": (16, 24)}
candidates = {name: [] for name in buckets}

for w in range(ytraj2.shape[0]):
    for r in range(n_regions):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        onset_h = int(np.argmax(lab))
        t_start = starts2[w]
        onset_abs_hour = t_start + WINDOW + onset_h
        for name, (lo, hi) in buckets.items():
            if lo <= onset_h < hi:
                peak_pm25 = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r].max()
                candidates[name].append((peak_pm25, w, r, onset_h, onset_abs_hour))

chosen = {}
used_onset_times = []  # onset_abs_hour of already-chosen examples (any region)

for name, cands in candidates.items():
    cands.sort(key=lambda x: -x[0])  # highest peak PM2.5 first
    for cand in cands:
        peak, w, r, onset_h, onset_abs_hour = cand
        conflict = any(abs(onset_abs_hour - used_t) < DEDUP_GAP_HOURS for used_t in used_onset_times)
        if not conflict:
            chosen[name] = cand
            used_onset_times.append(onset_abs_hour)
            break
    else:
        print(f"WARNING: no independent candidate found for {name} (try relaxing DEDUP_GAP_HOURS)")
        continue
    peak, w, r, onset_h, onset_abs_hour = chosen[name]
    print(f"{name}: window={w} region={region_names[r]} onset_hour={onset_h}  "
          f"peak_PM2.5={peak:.1f}  onset_time={dt_index[onset_abs_hour]}")

# ================================================================
# Run the trained decoder+localization model (p2) on these 3 examples
# and print + plot the trajectories
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
PROB_COLOR = "#2E5AAC"
PM25_COLOR = "#B3541E"

for ax, (name, (peak, w, r, onset_h, onset_abs_hour)) in zip(axes, chosen.items()):
    t_start = starts2[w]
    hours = np.arange(HORIZON)
    prob_traj = test_probs[w, r, :]  # test_probs already computed from p2 on the full test set
    label_traj = ytraj2[w, r, :]
    pm25_traj = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r]
    onset_dt = dt_index[onset_abs_hour]

    print(f"\n--- {name}: {region_names[r]}, forecast issued {dt_index[t_start + WINDOW - 1]}, "
          f"true onset at hour {onset_h} ({onset_dt}) ---")
    print("hour: " + " ".join(f"{h:5d}" for h in hours))
    print("prob: " + " ".join(f"{p:5.2f}" for p in prob_traj))
    print("PM25: " + " ".join(f"{p:5.0f}" for p in pm25_traj))
    print("label:" + " ".join(f"{int(l):5d}" for l in label_traj))

    ax2 = ax.twinx()
    ax.plot(hours, prob_traj, color=PROB_COLOR, linewidth=2, marker="o", markersize=3, label="predicted P(episode)")
    ax2.plot(hours, pm25_traj, color=PM25_COLOR, linewidth=1.5, alpha=0.8, marker="o", markersize=3, label="true PM2.5")
    ax2.axhline(75, color=PM25_COLOR, linestyle="--", linewidth=1, alpha=0.4)
    ax.axvline(onset_h, color="black", linewidth=1, alpha=0.6)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("P(episode)", color=PROB_COLOR)
    ax2.set_ylabel("PM2.5 (ug/m3)", color=PM25_COLOR)
    ax.set_title(f"{name}: {region_names[r]}, onset at hour {onset_h} ({onset_dt.strftime('%Y-%m-%d %H:%M')})")

axes[-1].set_xlabel("hours from forecast issue time")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{BASE}/fig_case_study_decoder_model.png", dpi=200)
plt.close()
print(f"\nsaved figure to {BASE}/fig_case_study_decoder_model.png")


early (0-8h): window=1360 region=Jeonbuk onset_hour=7  peak_PM2.5=195.6  onset_time=2019-02-28 11:00:00
middle (8-16h): window=253 region=Chungnam onset_hour=8  peak_PM2.5=128.0  onset_time=2019-01-13 09:00:00
late (16-24h): window=8214 region=Chungnam onset_hour=16  peak_PM2.5=120.0  onset_time=2019-12-11 10:00:00

--- early (0-8h): Jeonbuk, forecast issued 2019-02-28 03:00:00, true onset at hour 7 (2019-02-28 11:00:00) ---
hour:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23
prob:  0.98  0.91  0.89  0.86  0.84  0.82  0.79  0.77  0.75  0.74  0.73  0.72  0.70  0.69  0.68  0.68  0.68  0.67  0.67  0.66  0.67  0.68  0.68  0.68
PM25:    67    68    67    61    62    66    65    81    77    60    69    86   115   148   178   196   194   192   185   182   176   175   164   162
label:    0     0     0     0     0     0     0     1     0     0     0     1     1     1     1     1     1     1     1  

In [3]:
# ================================================================
# DIAGNOSTIC: distribution of onset_h (first hour with label=1 within
# the 24h horizon) across the TRAINING set, and specifically: for
# onset_h==0 cases, was the region ALREADY in a sustained episode at
# the last input hour (i.e., is this a continuing episode rather than
# a fresh onset)? This tests the "front-loaded onset times" hypothesis
# for why the localization loss learned a generic decay shape.
# ================================================================
import numpy as np

def onset_diagnostics(ytraj, starts, name):
    n_win, n_reg, H = ytraj.shape
    onset_hours = []
    already_ongoing = []
    for w in range(n_win):
        t_start = starts[w]
        last_input_hour = t_start + WINDOW - 1  # last hour of the INPUT window, i.e. "now"
        for r in range(n_reg):
            lab = ytraj[w, r, :]
            if lab.max() == 0:
                continue
            onset_h = int(np.argmax(lab))
            onset_hours.append(onset_h)
            if onset_h == 0:
                was_ongoing = region_episode_label[last_input_hour, r] > 0
                already_ongoing.append(bool(was_ongoing))

    onset_hours = np.array(onset_hours)
    already_ongoing = np.array(already_ongoing)

    print(f"\n{'='*20} {name}: onset_h distribution (n={len(onset_hours)} episode window/region pairs) {'='*20}")
    print(f"mean onset_h={onset_hours.mean():.2f}  median={np.median(onset_hours):.1f}")
    for lo, hi in [(0, 1), (0, 4), (0, 8), (8, 16), (16, 24)]:
        frac = ((onset_hours >= lo) & (onset_hours < hi)).mean()
        print(f"  fraction with onset_h in [{lo},{hi}): {frac:.1%}")

    n_onset0 = (onset_hours == 0).sum()
    print(f"\nonset_h==0 specifically: {n_onset0} cases ({n_onset0/len(onset_hours):.1%} of all episode windows)")
    if len(already_ongoing) > 0:
        print(f"  of these, already in a sustained episode at the last input hour (i.e. CONTINUING, not fresh): "
              f"{already_ongoing.mean():.1%}")
        print(f"  genuinely fresh onset exactly at hour 0 (episode starts right as window begins): "
              f"{(1-already_ongoing.mean()):.1%}")

onset_diagnostics(ytraj0, starts0, "TRAIN")
onset_diagnostics(ytraj1, starts1, "VAL")



==================== TRAIN: onset_h distribution (n=8356 episode window/region pairs) ====================
mean onset_h=8.41  median=7.0
  fraction with onset_h in [0,1): 26.5%
  fraction with onset_h in [0,4): 37.8%
  fraction with onset_h in [0,8): 51.5%
  fraction with onset_h in [8,16): 25.1%
  fraction with onset_h in [16,24): 23.4%

onset_h==0 specifically: 2216 cases (26.5% of all episode windows)
  of these, already in a sustained episode at the last input hour (i.e. CONTINUING, not fresh): 85.7%
  genuinely fresh onset exactly at hour 0 (episode starts right as window begins): 14.3%

==================== VAL: onset_h distribution (n=4377 episode window/region pairs) ====================
mean onset_h=8.04  median=6.0
  fraction with onset_h in [0,1): 28.1%
  fraction with onset_h in [0,4): 39.6%
  fraction with onset_h in [0,8): 53.9%
  fraction with onset_h in [8,16): 24.1%
  fraction with onset_h in [16,24): 22.0%

onset_h==0 specifically: 1229 cases (28.1% of all episode wi

In [1]:
# ================================================================
# LOSS-FUNCTION APPROACH v2: same decoder architecture and combined
# loss as before, but the localization cross-entropy now EXCLUDES
# "continuing" episodes (onset_h==0 AND the region was already in a
# sustained episode at the last input hour) -- confirmed to be 85.7%
# of all onset_h==0 cases and a trivial shortcut that was teaching
# the model a generic decay curve instead of genuine forecasting.
# Only genuinely fresh, forward-looking onsets now supervise the
# localization term.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
LOCALIZATION_WEIGHT = 0.3
MAX_EPOCHS_P2 = 15
PATIENCE = 5
RUN_TAG = f"localization_v2_w{LOCALIZATION_WEIGHT}"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN_Decoder(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local
        self.horizon = horizon

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)

        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_reg, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"device={DEVICE}  RUN_TAG={RUN_TAG}  LOCALIZATION_WEIGHT={LOCALIZATION_WEIGHT}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

# ---- NEW: precompute "already ongoing" mask (was the region already in a
# sustained episode at the LAST INPUT hour, i.e. immediately before the
# forecast horizon begins), per (window, region) ----
already_ongoing0 = (region_episode_label[starts0 + WINDOW - 1, :] > 0)  # (n_win0, n_regions)
already_ongoing1 = (region_episode_label[starts1 + WINDOW - 1, :] > 0)
already_ongoing0_t = torch.tensor(already_ongoing0)
already_ongoing1_t = torch.tensor(already_ongoing1)

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, starts, already_ongoing, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_bce, total_loc, total_n, total_loc_n = 0.0, 0.0, 0.0, 0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ao = already_ongoing[mb_idx].to(DEVICE)  # (mb, n_regions)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                bce = region_criterion(logits, yb)

                B_, R_, H_ = logits.shape
                logits_flat = logits.reshape(B_ * R_, H_)
                yb_flat = yb.reshape(B_ * R_, H_)
                ao_flat = ao.reshape(B_ * R_)
                has_ep = yb_flat.max(dim=-1).values > 0
                target_hour_all = yb_flat.argmax(dim=-1)
                # EXCLUDE continuing episodes: onset_h==0 AND already ongoing
                is_continuing = (target_hour_all == 0) & ao_flat
                valid_for_loc = has_ep & (~is_continuing)
                if valid_for_loc.any():
                    loc_loss = F.cross_entropy(logits_flat[valid_for_loc], target_hour_all[valid_for_loc])
                    n_loc = int(valid_for_loc.sum())
                else:
                    loc_loss = torch.tensor(0.0, device=DEVICE)
                    n_loc = 0

                loss = bce + LOCALIZATION_WEIGHT * loc_loss
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_bce += bce.item() * len(mb_idx)
            total_loc += float(loc_loss.detach()) * n_loc; total_loc_n += n_loc
            total_n += len(mb_idx)
        if train: optimizer.step()
    avg_loc = total_loc / max(total_loc_n, 1)
    return total_loss / max(total_n, 1), total_bce / max(total_n, 1), avg_loc

def predict_traj_probs(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} PHASE 1: station quantile pretraining (seed=0) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
p1 = StationQuantileGCN(n_feats, dropout=0.5).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=0.003, weight_decay=0.0005)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}")
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
del p1; gc.collect()

print(f"\n{'#'*15} PHASE 2: decoder + fixed localization loss (seed=0) {'#'*15}")
p2 = RegionFromTrajectoryGCN_Decoder(encoder_copy, region_membership_t, dropout=0.5).to(DEVICE)
opt2 = torch.optim.Adam(p2.parameters(), lr=0.003, weight_decay=0.0005)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
has_episode_val = ytraj1.max(axis=-1) > 0

best_val_aucpr, best_state, best_epoch = -1.0, None, 0
patience_counter = 0
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    tl, tbce, tloc = run_p2_traj_epoch(p2, X0_t, ytraj0_t, starts0_t, already_ongoing0_t, opt2, True)
    vl, vbce, vloc = run_p2_traj_epoch(p2, X1_t, ytraj1_t, starts1_t, already_ongoing1_t, opt2, False)
    val_probs = predict_traj_probs(p2, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    val_range = val_probs.max(axis=-1) - val_probs.min(axis=-1)
    val_flatness = float(np.median(val_range[has_episode_val]))
    print(f"  epoch {epoch}  train_loss={tl:.4f} (bce={tbce:.4f} loc={tloc:.4f})  "
          f"val_loss={vl:.4f} (bce={vbce:.4f} loc={vloc:.4f})  val_aucpr={va:.4f}  val_median_range={val_flatness:.4f}")
    if va > best_val_aucpr:
        best_val_aucpr, best_state, best_epoch = va, copy.deepcopy(p2.state_dict()), epoch
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (best was epoch {best_epoch})")
            break

p2.load_state_dict(best_state)
p2.eval()

test_probs = predict_traj_probs(p2, X2_t, starts2_t)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)
test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

has_episode = ytraj2.max(axis=-1) > 0
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} RESULTS: {RUN_TAG}, 2019 TEST (HORIZON={HORIZON}h, best_epoch={best_epoch}) {'='*20}")
print(f"val_AUCPR={best_val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
print(f"episode windows (n={has_episode.sum()}): prob range mean={prob_range[has_episode].mean():.4f}  median={np.median(prob_range[has_episode]):.4f}")
print(f"  (v1 localization loss, included continuing episodes, had: mean=0.2415 median=0.2224)")
print(f"  (original single-Linear-head model had: mean=0.2326 median=0.2020)")

with open(f"{BASE}/decoder_model_2019_{RUN_TAG}.json", "w") as f:
    json.dump({"run_tag": RUN_TAG, "localization_weight": LOCALIZATION_WEIGHT, "best_epoch": best_epoch,
               "val_aucpr": float(best_val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "prob_range_mean": float(prob_range[has_episode].mean()), "prob_range_median": float(np.median(prob_range[has_episode]))}, f, indent=2)
print(f"\nsaved to {BASE}/decoder_model_2019_{RUN_TAG}.json")


device=mps  RUN_TAG=localization_v2_w0.3  LOCALIZATION_WEIGHT=0.3
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

############### PHASE 1: station quantile pretraining (seed=0) ###############
  epoch 1  train_loss=0.1881  val_loss=0.1730
  epoch 2  train_loss=0.1808  val_loss=0.1747

############### PHASE 2: decoder + fixed localization loss (seed=0) ###############
  epoch 1  train_loss=1.2262 (bce=0.3873 loc=3.1829)  val_loss=0.4332 (bce=0.2923 loc=3.0727)  val_aucpr=0.5143  val_median_range=0.1106
  epoch 2  train_loss=1.1244 (bce=0.2788 loc=3.1267)  val_loss=0.4468 (bce=0.3091 loc=3.0291)  val_aucpr=0.5397  val_median_range=0.1448
  epoch 3  train_loss=1.0915 (bce=0.2546 loc=3.0951)  val_loss=0.4173 (bce=0.2821 loc=2.9876)  val_aucpr=0.5418  val_median_range=0.1706
  epoch 4  train_loss=1.0558 (bce=0.2402 loc=3.0567)  val_loss=0.4469 (bce=0.3085 loc=3.0680)  val_aucpr=0.5315  val_median_rang

In [2]:
# ================================================================
# CASE STUDY v2 (rerun on the FIXED localization-loss model):
# pick 3 genuinely distinct 24-hour episodes (onset times separated
# by at least 7 days), one each with onset early / middle / late in
# the 24h window. Uses p2 and test_probs already in memory from the
# v2 (continuing-episodes-excluded) training run.
# ================================================================
import numpy as np
import matplotlib.pyplot as plt

DEDUP_GAP_HOURS = 24 * 7  # require onset times at least 7 days apart

buckets = {"early (0-8h)": (0, 8), "middle (8-16h)": (8, 16), "late (16-24h)": (16, 24)}
candidates = {name: [] for name in buckets}

for w in range(ytraj2.shape[0]):
    for r in range(n_regions):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        onset_h = int(np.argmax(lab))
        t_start = starts2[w]
        onset_abs_hour = t_start + WINDOW + onset_h
        for name, (lo, hi) in buckets.items():
            if lo <= onset_h < hi:
                peak_pm25 = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r].max()
                candidates[name].append((peak_pm25, w, r, onset_h, onset_abs_hour))

chosen = {}
used_onset_times = []

for name, cands in candidates.items():
    cands.sort(key=lambda x: -x[0])
    for cand in cands:
        peak, w, r, onset_h, onset_abs_hour = cand
        conflict = any(abs(onset_abs_hour - used_t) < DEDUP_GAP_HOURS for used_t in used_onset_times)
        if not conflict:
            chosen[name] = cand
            used_onset_times.append(onset_abs_hour)
            break
    else:
        print(f"WARNING: no independent candidate found for {name}")
        continue
    peak, w, r, onset_h, onset_abs_hour = chosen[name]
    print(f"{name}: window={w} region={region_names[r]} onset_hour={onset_h}  "
          f"peak_PM2.5={peak:.1f}  onset_time={dt_index[onset_abs_hour]}")

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
PROB_COLOR = "#2E5AAC"
PM25_COLOR = "#B3541E"

for ax, (name, (peak, w, r, onset_h, onset_abs_hour)) in zip(axes, chosen.items()):
    t_start = starts2[w]
    hours = np.arange(HORIZON)
    prob_traj = test_probs[w, r, :]
    label_traj = ytraj2[w, r, :]
    pm25_traj = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r]
    onset_dt = dt_index[onset_abs_hour]

    print(f"\n--- {name}: {region_names[r]}, forecast issued {dt_index[t_start + WINDOW - 1]}, "
          f"true onset at hour {onset_h} ({onset_dt}) ---")
    print("hour: " + " ".join(f"{h:5d}" for h in hours))
    print("prob: " + " ".join(f"{p:5.2f}" for p in prob_traj))
    print("PM25: " + " ".join(f"{p:5.0f}" for p in pm25_traj))
    print("label:" + " ".join(f"{int(l):5d}" for l in label_traj))

    ax2 = ax.twinx()
    ax.plot(hours, prob_traj, color=PROB_COLOR, linewidth=2, marker="o", markersize=3, label="predicted P(episode)")
    ax2.plot(hours, pm25_traj, color=PM25_COLOR, linewidth=1.5, alpha=0.8, marker="o", markersize=3, label="true PM2.5")
    ax2.axhline(75, color=PM25_COLOR, linestyle="--", linewidth=1, alpha=0.4)
    ax.axvline(onset_h, color="black", linewidth=1, alpha=0.6)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("P(episode)", color=PROB_COLOR)
    ax2.set_ylabel("PM2.5 (ug/m3)", color=PM25_COLOR)
    ax.set_title(f"{name}: {region_names[r]}, onset at hour {onset_h} ({onset_dt.strftime('%Y-%m-%d %H:%M')})")

axes[-1].set_xlabel("hours from forecast issue time")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{BASE}/fig_case_study_decoder_v2.png", dpi=200)
plt.close()
print(f"\nsaved figure to {BASE}/fig_case_study_decoder_v2.png")


early (0-8h): window=1360 region=Jeonbuk onset_hour=7  peak_PM2.5=195.6  onset_time=2019-02-28 11:00:00
middle (8-16h): window=253 region=Chungnam onset_hour=8  peak_PM2.5=128.0  onset_time=2019-01-13 09:00:00
late (16-24h): window=8214 region=Chungnam onset_hour=16  peak_PM2.5=120.0  onset_time=2019-12-11 10:00:00

--- early (0-8h): Jeonbuk, forecast issued 2019-02-28 03:00:00, true onset at hour 7 (2019-02-28 11:00:00) ---
hour:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23
prob:  0.94  0.95  0.95  0.93  0.92  0.91  0.90  0.89  0.88  0.87  0.85  0.84  0.83  0.82  0.81  0.81  0.81  0.80  0.80  0.80  0.79  0.79  0.78  0.78
PM25:    67    68    67    61    62    66    65    81    77    60    69    86   115   148   178   196   194   192   185   182   176   175   164   162
label:    0     0     0     0     0     0     0     1     0     0     0     1     1     1     1     1     1     1     1  

In [1]:
# ================================================================
# BUCKET LOCALIZATION: keeps the exact same decoder architecture and
# full 24-hour probability trajectory output (objective 1, BCE,
# unchanged -- this is what gives you "yes/no in the next 24h" per
# region, already working well). ADDS a coarser objective 2: instead
# of asking the model to pinpoint the exact onset hour (which 5
# different attempts showed collapses to a generic population-average
# shape), it now only has to say which HALF of the window (hours 0-11
# vs 12-23) the onset falls in -- a much easier, more plausibly
# learnable estimation problem, while still narrowing a 24h risk
# window down to a 12h one.
#
# The bucket prediction is derived DIRECTLY from the same 24-hour
# logits the decoder already produces (max-pooled within each half),
# not a separate head -- so the trajectory output is untouched, this
# is purely an added loss term shaping how that trajectory behaves.
# Still excludes "continuing" episodes (onset_h==0 AND already
# elevated at the last input hour) from the bucket loss, per the
# earlier confirmed diagnostic.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
BUCKET_WEIGHT = 0.3
MAX_EPOCHS_P2 = 15
PATIENCE = 5
RUN_TAG = f"bucket2_w{BUCKET_WEIGHT}"

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM, HYBRID_THRESHOLD_KM = 250.0, 250.0, 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW = 36
HORIZON = 24
BUCKET_SPLIT = 12  # hours 0-11 = bucket 0 (early half), 12-23 = bucket 1 (late half)
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN_Decoder(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32, decoder_dim=16, horizon=HORIZON):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.decoder_pos_embed = nn.Parameter(torch.randn(horizon, decoder_dim) * 0.1)
        self.decoder_gru = nn.GRU(decoder_dim, gru_hidden, batch_first=True)
        self.output_head = nn.Linear(gru_hidden, 1)
        self.rmem = region_membership_t_local
        self.horizon = horizon

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)

        pos = self.decoder_pos_embed.unsqueeze(0).expand(B * n_reg, -1, -1)
        dec_out, _ = self.decoder_gru(pos, h_final)
        dec_out = self.drop(dec_out)
        logits = self.output_head(dec_out).squeeze(-1)
        return logits.reshape(B, n_reg, self.horizon)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1 = 2
print(f"device={DEVICE}  RUN_TAG={RUN_TAG}  BUCKET_WEIGHT={BUCKET_WEIGHT}  BUCKET_SPLIT={BUCKET_SPLIT}h")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

already_ongoing0 = (region_episode_label[starts0 + WINDOW - 1, :] > 0)
already_ongoing1 = (region_episode_label[starts1 + WINDOW - 1, :] > 0)
already_ongoing0_t = torch.tensor(already_ongoing0)
already_ongoing1_t = torch.tensor(already_ongoing1)

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")
region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, starts, already_ongoing, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_bce, total_bucket, total_n, total_bucket_n, total_bucket_correct = 0.0, 0.0, 0.0, 0, 0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ao = already_ongoing[mb_idx].to(DEVICE)
            ew_s = gather_seq(wind_ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                # ---- objective 1: full classification, unchanged ----
                bce = region_criterion(logits, yb)

                # ---- objective 2: coarse 2-bucket localization, derived
                # directly from the SAME 24h logits (max-pooled per half) ----
                B_, R_, H_ = logits.shape
                logits_flat = logits.reshape(B_ * R_, H_)
                yb_flat = yb.reshape(B_ * R_, H_)
                ao_flat = ao.reshape(B_ * R_)
                has_ep = yb_flat.max(dim=-1).values > 0
                target_hour_all = yb_flat.argmax(dim=-1)
                is_continuing = (target_hour_all == 0) & ao_flat
                valid = has_ep & (~is_continuing)

                if valid.any():
                    bucket_target = (target_hour_all[valid] >= BUCKET_SPLIT).long()  # 0=early half, 1=late half
                    early_logit = logits_flat[valid, :BUCKET_SPLIT].max(dim=-1).values
                    late_logit = logits_flat[valid, BUCKET_SPLIT:].max(dim=-1).values
                    bucket_logits = torch.stack([early_logit, late_logit], dim=-1)  # (n_valid, 2)
                    bucket_loss = F.cross_entropy(bucket_logits, bucket_target)
                    n_valid = int(valid.sum())
                    bucket_correct = int((bucket_logits.argmax(dim=-1) == bucket_target).sum())
                else:
                    bucket_loss = torch.tensor(0.0, device=DEVICE)
                    n_valid = 0
                    bucket_correct = 0

                loss = bce + BUCKET_WEIGHT * bucket_loss
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_bce += bce.item() * len(mb_idx)
            total_bucket += float(bucket_loss.detach()) * n_valid
            total_bucket_n += n_valid
            total_bucket_correct += bucket_correct
            total_n += len(mb_idx)
        if train: optimizer.step()
    bucket_acc = total_bucket_correct / max(total_bucket_n, 1)
    return (total_loss / max(total_n, 1), total_bce / max(total_n, 1),
            total_bucket / max(total_bucket_n, 1), bucket_acc)

def predict_traj_probs(model, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(wind_ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(wind_ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

print(f"\n{'#'*15} PHASE 1: station quantile pretraining (seed=0) {'#'*15}")
torch.manual_seed(0); np.random.seed(0)
p1 = StationQuantileGCN(n_feats, dropout=0.5).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=0.003, weight_decay=0.0005)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    tl = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    print(f"  epoch {epoch}  train_loss={tl:.4f}  val_loss={vl:.4f}")
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
del p1; gc.collect()

print(f"\n{'#'*15} PHASE 2: decoder + 2-bucket localization loss (seed=0) {'#'*15}")
p2 = RegionFromTrajectoryGCN_Decoder(encoder_copy, region_membership_t, dropout=0.5).to(DEVICE)
opt2 = torch.optim.Adam(p2.parameters(), lr=0.003, weight_decay=0.0005)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
has_episode_val = ytraj1.max(axis=-1) > 0

best_val_aucpr, best_state, best_epoch = -1.0, None, 0
patience_counter = 0
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    tl, tbce, tbucket, tbucket_acc = run_p2_traj_epoch(p2, X0_t, ytraj0_t, starts0_t, already_ongoing0_t, opt2, True)
    vl, vbce, vbucket, vbucket_acc = run_p2_traj_epoch(p2, X1_t, ytraj1_t, starts1_t, already_ongoing1_t, opt2, False)
    val_probs = predict_traj_probs(p2, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    val_range = val_probs.max(axis=-1) - val_probs.min(axis=-1)
    val_flatness = float(np.median(val_range[has_episode_val]))
    print(f"  epoch {epoch}  train (bce={tbce:.4f} bucket={tbucket:.4f} bucket_acc={tbucket_acc:.3f})  "
          f"val (bce={vbce:.4f} bucket={vbucket:.4f} bucket_acc={vbucket_acc:.3f})  "
          f"val_aucpr={va:.4f}  val_median_range={val_flatness:.4f}")
    if va > best_val_aucpr:
        best_val_aucpr, best_state, best_epoch = va, copy.deepcopy(p2.state_dict()), epoch
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  early stopping at epoch {epoch} (best was epoch {best_epoch})")
            break

p2.load_state_dict(best_state)
p2.eval()

test_probs = predict_traj_probs(p2, X2_t, starts2_t)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)
test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

# ---- final test-set bucket accuracy, for the headline metric ----
already_ongoing2 = (region_episode_label[starts2 + WINDOW - 1, :] > 0)
n_win2, n_reg2, H2 = ytraj2.shape
bucket_correct, bucket_total = 0, 0
for w in range(n_win2):
    for r in range(n_reg2):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        onset_h = int(np.argmax(lab))
        if onset_h == 0 and already_ongoing2[w, r]:
            continue  # exclude continuing episodes, same as training
        true_bucket = 1 if onset_h >= BUCKET_SPLIT else 0
        prob = test_probs[w, r, :]
        pred_bucket = 1 if prob[BUCKET_SPLIT:].max() > prob[:BUCKET_SPLIT].max() else 0
        bucket_correct += int(pred_bucket == true_bucket)
        bucket_total += 1

bucket_test_acc = bucket_correct / max(bucket_total, 1)

has_episode = ytraj2.max(axis=-1) > 0
prob_range = test_probs.max(axis=-1) - test_probs.min(axis=-1)

print(f"\n{'='*20} RESULTS: {RUN_TAG}, 2019 TEST (HORIZON={HORIZON}h, best_epoch={best_epoch}) {'='*20}")
print(f"val_AUCPR={best_val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
print(f"TEST 2-bucket (early vs late half) accuracy: {bucket_test_acc:.4f}  (n={bucket_total}, chance=0.50)")
print(f"episode windows (n={has_episode.sum()}): prob range mean={prob_range[has_episode].mean():.4f}  median={np.median(prob_range[has_episode]):.4f}")

with open(f"{BASE}/decoder_model_2019_{RUN_TAG}.json", "w") as f:
    json.dump({"run_tag": RUN_TAG, "bucket_weight": BUCKET_WEIGHT, "bucket_split_hour": BUCKET_SPLIT,
               "best_epoch": best_epoch, "val_aucpr": float(best_val_aucpr), "test_aucroc": float(test_aucroc),
               "test_aucpr": float(test_aucpr), "bucket_test_accuracy": float(bucket_test_acc),
               "prob_range_mean": float(prob_range[has_episode].mean()),
               "prob_range_median": float(np.median(prob_range[has_episode]))}, f, indent=2)
print(f"\nsaved to {BASE}/decoder_model_2019_{RUN_TAG}.json")


device=mps  RUN_TAG=bucket2_w0.3  BUCKET_WEIGHT=0.3  BUCKET_SPLIT=12h
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17485 val=8701 test=8701
pos_weight: 50.00

############### PHASE 1: station quantile pretraining (seed=0) ###############
  epoch 1  train_loss=0.1881  val_loss=0.1730
  epoch 2  train_loss=0.1808  val_loss=0.1747

############### PHASE 2: decoder + 2-bucket localization loss (seed=0) ###############
  epoch 1  train (bce=0.3820 bucket=0.6684 bucket_acc=0.602)  val (bce=0.2959 bucket=0.6231 bucket_acc=0.690)  val_aucpr=0.5229  val_median_range=0.1120
  epoch 2  train (bce=0.2679 bucket=0.6356 bucket_acc=0.653)  val (bce=0.3065 bucket=0.5790 bucket_acc=0.717)  val_aucpr=0.5519  val_median_range=0.1470
  epoch 3  train (bce=0.2495 bucket=0.6163 bucket_acc=0.661)  val (bce=0.3035 bucket=0.6027 bucket_acc=0.702)  val_aucpr=0.5341  val_median_range=0.1810
  epoch 4  train (bce=0.2286 bucket=0.5848 bucket_acc=0.687)  val (bce=0.

In [2]:
# ================================================================
# CASE STUDY v3 (rerun on the bucket-loss model): pick 3 genuinely
# distinct 24-hour episodes (onset times separated by at least 7 days),
# one each with onset early / middle / late in the 24h window. Uses
# p2 and test_probs already in memory from the bucket2_w0.3 run.
# ================================================================
import numpy as np
import matplotlib.pyplot as plt

DEDUP_GAP_HOURS = 24 * 7

buckets = {"early (0-8h)": (0, 8), "middle (8-16h)": (8, 16), "late (16-24h)": (16, 24)}
candidates = {name: [] for name in buckets}

for w in range(ytraj2.shape[0]):
    for r in range(n_regions):
        lab = ytraj2[w, r, :]
        if lab.max() == 0:
            continue
        onset_h = int(np.argmax(lab))
        t_start = starts2[w]
        onset_abs_hour = t_start + WINDOW + onset_h
        for name, (lo, hi) in buckets.items():
            if lo <= onset_h < hi:
                peak_pm25 = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r].max()
                candidates[name].append((peak_pm25, w, r, onset_h, onset_abs_hour))

chosen = {}
used_onset_times = []

for name, cands in candidates.items():
    cands.sort(key=lambda x: -x[0])
    for cand in cands:
        peak, w, r, onset_h, onset_abs_hour = cand
        conflict = any(abs(onset_abs_hour - used_t) < DEDUP_GAP_HOURS for used_t in used_onset_times)
        if not conflict:
            chosen[name] = cand
            used_onset_times.append(onset_abs_hour)
            break
    else:
        print(f"WARNING: no independent candidate found for {name}")
        continue
    peak, w, r, onset_h, onset_abs_hour = chosen[name]
    true_bucket = "late" if onset_h >= 12 else "early"
    print(f"{name}: window={w} region={region_names[r]} onset_hour={onset_h} (TRUE bucket={true_bucket})  "
          f"peak_PM2.5={peak:.1f}  onset_time={dt_index[onset_abs_hour]}")

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
PROB_COLOR = "#2E5AAC"
PM25_COLOR = "#B3541E"

for ax, (name, (peak, w, r, onset_h, onset_abs_hour)) in zip(axes, chosen.items()):
    t_start = starts2[w]
    hours = np.arange(HORIZON)
    prob_traj = test_probs[w, r, :]
    label_traj = ytraj2[w, r, :]
    pm25_traj = region_pm25[t_start + WINDOW: t_start + WINDOW + HORIZON, r]
    onset_dt = dt_index[onset_abs_hour]

    early_max, late_max = prob_traj[:12].max(), prob_traj[12:].max()
    pred_bucket = "late" if late_max > early_max else "early"
    true_bucket = "late" if onset_h >= 12 else "early"

    print(f"\n--- {name}: {region_names[r]}, forecast issued {dt_index[t_start + WINDOW - 1]}, "
          f"true onset at hour {onset_h} ({onset_dt}), TRUE bucket={true_bucket}, PRED bucket={pred_bucket} "
          f"(early_max={early_max:.2f}, late_max={late_max:.2f}) ---")
    print("hour: " + " ".join(f"{h:5d}" for h in hours))
    print("prob: " + " ".join(f"{p:5.2f}" for p in prob_traj))
    print("PM25: " + " ".join(f"{p:5.0f}" for p in pm25_traj))
    print("label:" + " ".join(f"{int(l):5d}" for l in label_traj))

    ax2 = ax.twinx()
    ax.plot(hours, prob_traj, color=PROB_COLOR, linewidth=2, marker="o", markersize=3, label="predicted P(episode)")
    ax2.plot(hours, pm25_traj, color=PM25_COLOR, linewidth=1.5, alpha=0.8, marker="o", markersize=3, label="true PM2.5")
    ax2.axhline(75, color=PM25_COLOR, linestyle="--", linewidth=1, alpha=0.4)
    ax.axvline(onset_h, color="black", linewidth=1, alpha=0.6)
    ax.axvline(12, color="gray", linewidth=1.5, linestyle=":", alpha=0.7, label="bucket split")

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("P(episode)", color=PROB_COLOR)
    ax2.set_ylabel("PM2.5 (ug/m3)", color=PM25_COLOR)
    ax.set_title(f"{name}: {region_names[r]}, true={true_bucket} / pred={pred_bucket}  "
                 f"(onset hour {onset_h}, {onset_dt.strftime('%Y-%m-%d %H:%M')})")

axes[-1].set_xlabel("hours from forecast issue time")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{BASE}/fig_case_study_bucket_model.png", dpi=200)
plt.close()
print(f"\nsaved figure to {BASE}/fig_case_study_bucket_model.png")


early (0-8h): window=1360 region=Jeonbuk onset_hour=7 (TRUE bucket=early)  peak_PM2.5=195.6  onset_time=2019-02-28 11:00:00
middle (8-16h): window=253 region=Chungnam onset_hour=8 (TRUE bucket=early)  peak_PM2.5=128.0  onset_time=2019-01-13 09:00:00
late (16-24h): window=8214 region=Chungnam onset_hour=16 (TRUE bucket=late)  peak_PM2.5=120.0  onset_time=2019-12-11 10:00:00

--- early (0-8h): Jeonbuk, forecast issued 2019-02-28 03:00:00, true onset at hour 7 (2019-02-28 11:00:00), TRUE bucket=early, PRED bucket=early (early_max=0.96, late_max=0.90) ---
hour:     0     1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17    18    19    20    21    22    23
prob:  0.95  0.96  0.96  0.95  0.95  0.96  0.96  0.96  0.96  0.95  0.93  0.92  0.90  0.90  0.88  0.86  0.87  0.87  0.86  0.86  0.85  0.85  0.82  0.81
PM25:    67    68    67    61    62    66    65    81    77    60    69    86   115   148   178   196   194   192   185   182   176   175   16